# Contracts Extraction — Redash / Trino → Parquet

Pulls contract-related data from Trino (CLM contract terms, plus fact-table
reconciliations that lean on those terms — satellite fees, KAM fees, defective returns,
recall assistance, early payment, fill rate, storage fees, RTV days, ads spend, turnover
discount, ePOD, SP benchmarking, PRN quantity, city/warehouse distance) and writes **one
Parquet file per query** — 16 in total — for downstream AI-generated reporting on a web
platform.

**This revision:**
- **Parquet output** — each pipeline streams its fetched batches straight into one
  `pyarrow.parquet.ParquetWriter` against a pinned schema (built from each pipeline's
  column list + declared numeric/date columns), so every batch's dtypes line up and the
  file is directly queryable/loadable (`pd.read_parquet` / `pyarrow.parquet.read_table`)
  by the downstream reporting layer — no CSV round-trip, no dtype guessing.
- **One pass, not two** — data-quality stats (row counts, null counts) are accumulated
  *while* streaming each batch, instead of a second full read-back over the output
  afterward.
- **Shared runner functions** — one `run_id_batched()`, `run_date_batched()`, and
  `run_month_batched()` function each handle the batching/fetch/write/log/stats loop for
  every pipeline of that kind (plus `run_single()` for the one small, non-batched
  reference query), instead of 16 near-identical copies of the same loop. Each pipeline
  is now just its SQL text + a column spec + one function call.
- **Timestamped logs** — a small `log()` helper prints `[HH:MM:SS] [label] message` lines
  so a run's progress (and where it's stuck, if it's stuck) is visible as it goes.
- **100MB-per-file cap** — every pipeline checks its current output file's on-disk size
  after each batch and automatically rolls over to a new numbered part
  (`..._part001.parquet`, `..._part002.parquet`, ...) once it hits `MAX_FILE_MB` (default
  100) in the CONFIG cell, rather than growing one file indefinitely. A query's full
  output is the *set* of every part file sharing its base name.
- **Per-query Yes/No run flags** — a `RUN_FLAGS` dict in the CONFIG cell lets you switch
  individual queries on/off before running the notebook top-to-bottom, without deleting or
  commenting out any code. Skipped queries are logged and marked `SKIPPED` in the final
  Output Summary rather than silently vanishing.

**One shared period, `START_DATE`/`END_DATE` in the CONFIG cell — applies to 6 of the 16
queries, not all of them:**

| Kind | Behavior | Queries |
|---|---|---|
| **Period-scoped** | Filtered to `START_DATE`..`END_DATE`, batched by date window (or by month for the one query parameterized that way) | Satellite City Fee, BCPL Defective CRBS, Recall Assistance, Early Payment Discount, Fill Rate Penalty, PRN Monthly by Entity |
| **Current-snapshot** | Always pulls the *current* state of active contracts — no date filter in the source query | Contracts, Satellite Fees Terms, KAM Fees, Storage Fees, RTV Days, Ads Spend, Turnover Discount, ePOD Clause, SP Benchmarking |
| **Reference (not batched)** | Small lookup table, no date filter, run once | City/Warehouse Distance & Satellite Classification |

That split reflects what each source query actually filters on, not a simplification —
the current-snapshot queries batch by `contract_id` range instead (nothing to window on
by date); the period-scoped ones batch by date window or month.

**How to run:**
1. In the CONFIG cell, set `START_DATE`/`END_DATE` (and other knobs, if needed).
2. In the same cell, set each entry in `RUN_FLAGS` to `'Yes'` or `'No'` depending on which
   of the 16 queries you actually want to run this time (e.g. set everything to `'No'`
   except the one query you're testing, to iterate quickly without waiting on the other 15).
3. Run every cell top-to-bottom. Queries flagged `'No'` are skipped instantly (no query
   sent to Trino) and logged as such.

The final section (Output Summary) lists every query, whether it ran or was skipped, its
row count, and file size.

See `queries_repository.md` (next to this notebook) for the full history/context/decisions
behind every individual query.

## 1. CONFIG — edit knobs, then run everything top-to-bottom

In [ ]:
# ============================================================================
# CONFIG
# ============================================================================
import os

# ---- Reporting period --------------------------------------------------------
# Drives 6 of the 15 queries (see table above); the other 9 always reflect the
# current state of active contracts regardless of this window.
START_DATE = '2026-04-01'
END_DATE   = '2026-06-30'

# ---- Query-specific params ----------------------------------------------------
DAY_WEEK_MONTH = 'day'   # granularity for queries with a {{day_week_month}} placeholder: 'day' | 'week' | 'month'
MFR_ID   = '%%'          # '%%' = all manufacturers; else exact manufacturer_id, e.g. '40'
MFR_NAME = '%%'          # '%%' = all manufacturers; else case-insensitive substring match

# PRN Monthly is parameterized by a single {{target_month}} rather than a range; None
# derives the month list from START_DATE..END_DATE, or pass an explicit list like
# ['2025-04','2025-05'] to override.
PRN_MONTHLY_TARGET_MONTHS = None

# ---- Run flags ------------------------------------------------------------------
# Set each query to 'Yes' to run it this pass, or 'No' to skip it entirely (no query is
# sent to Trino for a skipped pipeline - it's just logged as SKIPPED and left out of the
# Output Summary's row counts). Handy for testing one query at a time instead of waiting
# on all 15 every run. Keys must match PARQUET_FILES keys below.
RUN_FLAGS = {
    'contracts':           'No',
    'satellite_city_list': 'Yes',   
    'satellite_city_fee':  'No',   
    'kam_fees':            'No',
    'bcpl_defective_crbs': 'No',
    'recall_assistance':   'No',
    'early_payment':       'No',
    'fill_rate_penalty':   'No',
    'storage_fees':        'No',
    'rtv_days':            'No',
    'ads_spend':           'No',
    'turnover_discount':   'No',
    'epod_clause':         'No',
    'sp_benchmarking':     'No',
    'prn_monthly':         'No',
    'city_distance':       'No',
}



def should_run(label):
    """True unless RUN_FLAGS[label] is explicitly 'No' (case/whitespace-insensitive)."""
    flag = str(RUN_FLAGS.get(label, 'Yes')).strip().lower()
    if flag not in ('yes', 'no'):
        raise ValueError(f"RUN_FLAGS['{label}'] = {RUN_FLAGS.get(label)!r} - must be 'Yes' or 'No'")
    return flag == 'yes'


# ---- Output -------------------------------------------------------------------
OUT_DIR = os.path.expanduser('~/contracts_extract')


def _ensure_writable(preferred):
    """Return the first directory we can actually create + write into."""
    candidates = [preferred, os.path.join(os.getcwd(), 'contracts_extract')]
    for cand in candidates:
        try:
            os.makedirs(cand, exist_ok=True)
            probe = os.path.join(cand, '.write_test')
            with open(probe, 'w') as f:
                f.write('ok')
            os.remove(probe)
            return cand
        except (PermissionError, OSError) as e:
            print(f'  not writable: {cand}  ({e})')
    raise PermissionError(
        'No writable output dir. Set OUT_DIR to a path you own and re-run this cell.'
    )


OUT_DIR = _ensure_writable(OUT_DIR)

PARQUET_FILES = {
    'contracts':           os.path.join(OUT_DIR, 'contracts_extract.parquet'),
    'satellite_city_list': os.path.join(OUT_DIR, 'satellite_city_list.parquet'),
    'satellite_city_fee':  os.path.join(OUT_DIR, 'satellite_city_fee.parquet'),
    'kam_fees':            os.path.join(OUT_DIR, 'kam_fees_extract.parquet'),
    'bcpl_defective_crbs': os.path.join(OUT_DIR, 'bcpl_defective_crbs.parquet'),
    'recall_assistance':   os.path.join(OUT_DIR, 'recall_assistance_prn.parquet'),
    'early_payment':       os.path.join(OUT_DIR, 'early_payment_discount.parquet'),
    'fill_rate_penalty':   os.path.join(OUT_DIR, 'fill_rate_penalty.parquet'),
    'storage_fees':        os.path.join(OUT_DIR, 'storage_fees_extract.parquet'),
    'rtv_days':            os.path.join(OUT_DIR, 'rtv_days_extract.parquet'),
    'ads_spend':           os.path.join(OUT_DIR, 'ads_spend_extract.parquet'),
    'turnover_discount':   os.path.join(OUT_DIR, 'turnover_discount_extract.parquet'),
    'epod_clause':         os.path.join(OUT_DIR, 'epod_clause_extract.parquet'),
    'sp_benchmarking':     os.path.join(OUT_DIR, 'sp_benchmarking_extract.parquet'),
    'prn_monthly':         os.path.join(OUT_DIR, 'prn_monthly_by_entity.parquet'),
    'city_distance':       os.path.join(OUT_DIR, 'city_warehouse_distance.parquet'),
}


# ---- Performance / memory knobs ------------------------------------------------
CHUNKSIZE = 100_000   # rows pulled per fetch within a batch; lower if memory is tight

# contract_id batching (current-snapshot queries)
BATCH_SIZE_CONTRACTS  = 5_000
BATCH_SIZE_SATELLITE  = 5_000
BATCH_SIZE_KAM        = 5_000
BATCH_SIZE_STORAGE    = 5_000
BATCH_SIZE_RTV        = 5_000
BATCH_SIZE_ADSPEND    = 5_000
BATCH_SIZE_TURNOVER   = 5_000
BATCH_SIZE_EPOD       = 5_000
BATCH_SIZE_SPBENCH    = 5_000

# date-window batching (period-scoped queries), days per window
BATCH_DAYS_SATCITYFEE = 7
BATCH_DAYS_SATCITYLIST = 30   # NEW: date window for satellite city list
BATCH_DAYS_BCPL       = 7
BATCH_DAYS_RECALL     = 7
BATCH_DAYS_EARLYPAY   = 7
BATCH_DAYS_FILLRATE   = 7

# ---- File size cap --------------------------------------------------------------
# No single output Parquet file is allowed to exceed this size. Each pipeline checks its
# on-disk file size after every batch and, once it's at/over this cap, closes that file
# and rolls over to a new numbered part (e.g. contracts_extract_part001.parquet,
# contracts_extract_part002.parquet, ...) rather than growing one file indefinitely. A
# query's full output is therefore the *set* of every part file sharing its base name -
# read them together downstream with pd.concat([pd.read_parquet(f) for f in glob(...)])
# or pyarrow.dataset.dataset(directory, format='parquet') / duckdb reading a glob pattern.
MAX_FILE_MB = 100

# Threshold (km) used to classify a city as "satellite" for query 16 - matches the
# > 100km convention already used in the IOCC Satellite City Fee query (query 3).
SATELLITE_DISTANCE_KM_THRESHOLD = 100

print(f'Period                : {START_DATE} -> {END_DATE}  (grain={DAY_WEEK_MONTH}, Mfr ID={MFR_ID}, Mfr Name={MFR_NAME})')
print(f'Output dir            : {OUT_DIR}')
print(f'Max file size         : {MAX_FILE_MB} MB per part (files auto-split beyond this)')
print(f'Queries to run ({sum(1 for k in RUN_FLAGS if should_run(k))}/{len(RUN_FLAGS)}):')
for k, v in PARQUET_FILES.items():
    print(f"    {k:<20} run={RUN_FLAGS.get(k, 'Yes'):<3}  {v}")

## 2. Imports, connection, logging & helpers

In [ ]:
# ============================================================================
# Imports + connection + logging + helpers
# ============================================================================
import re
import time
import datetime as dt
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import pencilbox as pb

connection = pb.get_connection('[Warehouse] Trino')


def log(msg, label=None):
    """Timestamped progress line: [HH:MM:SS] [label] message."""
    ts = dt.datetime.now().strftime('%H:%M:%S')
    tag = f'[{label}] ' if label else ''
    print(f'[{ts}] {tag}{msg}')


def render(query, **params):
    """Replace Redash-style {{placeholders}} (whitespace-tolerant) with values."""
    out = query
    for key, val in params.items():
        out = re.sub(r'\{\{\s*' + re.escape(key) + r'\s*\}\}', str(val), out)
    leftover = re.findall(r'\{\{.*?\}\}', out)
    if leftover:
        raise ValueError(f'Unsubstituted placeholders remain: {sorted(set(leftover))}')
    return out


def sql(query, label=None):
    """Run a query, return a DataFrame, log row count + elapsed time."""
    t0 = time.time()
    df = pd.read_sql_query(sql=query, con=connection)
    log(f'{time.time()-t0:6.1f}s  {len(df):>10,} rows', label=label)
    return df


def _iter_chunks(query):
    """Yield DataFrame chunks; fall back to one fetch if chunksize unsupported."""
    try:
        for chunk in pd.read_sql_query(sql=query, con=connection, chunksize=CHUNKSIZE):
            yield chunk
    except TypeError:
        yield pd.read_sql_query(sql=query, con=connection)


def id_windows(min_id, max_id, step):
    """Split an inclusive [min_id, max_id] contract_id range into consecutive
    step-wide windows."""
    out, cur = [], int(min_id)
    max_id = int(max_id)
    while cur <= max_id:
        w_end = min(cur + step - 1, max_id)
        out.append((cur, w_end))
        cur = w_end + 1
    return out


def date_windows(start_str, end_str, step_days):
    """Split an inclusive [start, end] date range into consecutive step_days windows."""
    from datetime import date, timedelta
    s = date.fromisoformat(start_str)
    e = date.fromisoformat(end_str)
    out, cur = [], s
    while cur <= e:
        w_end = min(cur + timedelta(days=step_days - 1), e)
        out.append((cur.isoformat(), w_end.isoformat()))
        cur = w_end + timedelta(days=1)
    return out


def month_range(start_str, end_str):
    """Return a list of 'YYYY-MM' strings covering every calendar month overlapping
    [start_str, end_str]."""
    from datetime import date
    s = date.fromisoformat(start_str)
    e = date.fromisoformat(end_str)
    months, y, m = [], s.year, s.month
    while (y, m) <= (e.year, e.month):
        months.append(f'{y:04d}-{m:02d}')
        m += 1
        if m > 12:
            m = 1
            y += 1
    return months


def build_schema(columns, numeric_cols=(), date_cols=()):
    """Pin one Arrow schema per pipeline (float64 for numeric_cols, timestamp('us') for
    date_cols, string for everything else) so every batch written to the same Parquet
    file - across potentially hundreds of batches - has identical column types. Without
    this, a batch that happens to be all-null in some column could get its dtype inferred
    differently than an earlier batch, and ParquetWriter would reject the mismatch."""
    fields = []
    for c in columns:
        if c in numeric_cols:
            fields.append(pa.field(c, pa.float64()))
        elif c in date_cols:
            fields.append(pa.field(c, pa.timestamp('us')))
        else:
            fields.append(pa.field(c, pa.string()))
    return pa.schema(fields)


def normalize_chunk(df, columns, numeric_cols=(), date_cols=(), stamp=None):
    """Reindex a raw fetched chunk to a fixed column order and coerce dtypes so every
    batch written to the same Parquet file has consistent columns/typing. `stamp`
    optionally adds/overwrites a column with a constant value (e.g. target_month) not
    present in the query's own output."""
    df = df.loc[:, ~df.columns.duplicated()].copy()
    if stamp:
        for col, val in stamp.items():
            df[col] = val
    df = df.reindex(columns=columns)
    for c in numeric_cols:
        df[c] = pd.to_numeric(df[c], errors='coerce')
    for c in date_cols:
        df[c] = pd.to_datetime(df[c], errors='coerce')
    str_cols = [c for c in columns if c not in numeric_cols and c not in date_cols]
    for c in str_cols:
        s = df[c].astype('object').where(df[c].notna(), None)
        df[c] = s.map(lambda v: None if v is None else str(v))
    return df

## 3. Generic batched-runner functions

One function per batching strategy. Each fetches in `CHUNKSIZE`-row pieces per batch,
normalizes, appends straight to the current Parquet part (rolling to a new part once
  MAX_FILE_MB is reached
row group), logs progress per batch, and accumulates row/null counts in the same pass —
no second read-back needed for the quality check.

In [ ]:
import glob


def part_path(base_path, part_num):
    """contracts_extract.parquet, part 3 -> contracts_extract_part003.parquet"""
    root, ext = os.path.splitext(base_path)
    return f'{root}_part{part_num:03d}{ext}'


def _clear_existing_parts(base_path):
    """Remove every part file (and the bare pre-split filename, defensively) from a
    prior run before starting a fresh one, so stale parts never linger alongside new
    ones."""
    root, ext = os.path.splitext(base_path)
    for f in glob.glob(f'{root}_part*{ext}'):
        os.remove(f)
    if os.path.exists(base_path):
        os.remove(base_path)


def _open_new_part(writer_state, schema):
    path = part_path(writer_state['base_path'], writer_state['part'])
    writer_state['writer'] = pq.ParquetWriter(path, schema)
    writer_state['current_path'] = path
    writer_state['paths'].append(path)


def _write_and_track(df, schema, writer_state, stats):
    """Convert one normalized batch to the pipeline's pinned Arrow schema, append it to
    the current part's ParquetWriter (opening a new part first if none is open yet),
    accumulate row/null stats in the same pass, then roll over to a new part if this
    part's on-disk size has reached MAX_FILE_MB."""
    if writer_state['writer'] is None:
        _open_new_part(writer_state, schema)

    table = pa.Table.from_pandas(df, schema=schema, preserve_index=False)
    writer_state['writer'].write_table(table)
    stats['rows'] += len(df)
    for c in df.columns:
        stats['nulls'][c] = stats['nulls'].get(c, 0) + int(df[c].isna().sum())

    size_mb = os.path.getsize(writer_state['current_path']) / (1024 * 1024)
    if size_mb >= MAX_FILE_MB:
        writer_state['writer'].close()
        writer_state['writer'] = None
        writer_state['part'] += 1


def _close_writer(writer_state):
    if writer_state['writer'] is not None:
        writer_state['writer'].close()
        writer_state['writer'] = None


def _new_writer_state(base_path):
    return {'writer': None, 'base_path': base_path, 'part': 1, 'paths': [], 'current_path': None}


def run_id_batched(label, sql_template, columns, parquet_path, bounds_sql, batch_size,
                     numeric_cols=(), date_cols=()):
    """Batches by contract_id range. Used for current-snapshot contract-terms queries.
    Output may be split across multiple part files if it exceeds MAX_FILE_MB - see
    part_path()/PARQUET_FILES's docstring in CONFIG."""
    log('starting (contract_id batched)', label=label)
    _clear_existing_parts(parquet_path)
    schema = build_schema(columns, numeric_cols, date_cols)
    bounds = sql(bounds_sql, label=f'{label}-bounds')
    min_id, max_id, n_contracts = bounds.loc[0, ['min_id', 'max_id', 'n_contracts']]
    windows = id_windows(min_id, max_id, batch_size)
    log(f'contract_id range {min_id}->{max_id} ({n_contracts:,} contracts) -> {len(windows)} batch(es)', label=label)

    writer_state = _new_writer_state(parquet_path)
    stats = {'rows': 0, 'nulls': {}}
    t0 = time.time()
    for i, (id_from, id_to) in enumerate(windows, 1):
        q = render(sql_template, id_from=id_from, id_to=id_to)
        batch_rows = 0
        for chunk in _iter_chunks(q):
            if len(chunk) == 0:
                continue
            df = normalize_chunk(chunk, columns, numeric_cols, date_cols)
            _write_and_track(df, schema, writer_state, stats)
            batch_rows += len(df)
        log(f'batch {i}/{len(windows)}  ids {id_from}..{id_to}: {batch_rows:>8,} rows  (cum {stats["rows"]:>10,}, {len(writer_state["paths"])} part(s) so far)', label=label)
    _close_writer(writer_state)
    log(f'DONE {time.time()-t0:6.1f}s  {stats["rows"]:,} rows -> {len(writer_state["paths"])} part file(s), base {parquet_path}', label=label)
    return {'label': label, 'base_path': parquet_path, 'files': writer_state['paths'], 'period_scoped': False, **stats}


def run_date_batched(label, sql_template, columns, parquet_path, batch_days,
                       numeric_cols=(), date_cols=(), extra_params=None):
    """Batches by START_DATE..END_DATE date window. Used for period-scoped queries hitting
    transaction-level fact tables. Output may be split across multiple part files if it
    exceeds MAX_FILE_MB - see part_path()/PARQUET_FILES's docstring in CONFIG."""
    log('starting (date-window batched)', label=label)
    _clear_existing_parts(parquet_path)
    schema = build_schema(columns, numeric_cols, date_cols)
    windows = date_windows(START_DATE, END_DATE, batch_days)
    log(f'{START_DATE} -> {END_DATE} -> {len(windows)} batch(es) of up to {batch_days} day(s)', label=label)

    writer_state = _new_writer_state(parquet_path)
    stats = {'rows': 0, 'nulls': {}}
    t0 = time.time()
    for i, (w_from, w_to) in enumerate(windows, 1):
        params = dict(extra_params or {})
        params.update({'start_date': w_from, 'end_date': w_to})
        q = render(sql_template, **params)
        batch_rows = 0
        for chunk in _iter_chunks(q):
            if len(chunk) == 0:
                continue
            df = normalize_chunk(chunk, columns, numeric_cols, date_cols)
            _write_and_track(df, schema, writer_state, stats)
            batch_rows += len(df)
        log(f'window {i}/{len(windows)}  {w_from}..{w_to}: {batch_rows:>8,} rows  (cum {stats["rows"]:>10,}, {len(writer_state["paths"])} part(s) so far)', label=label)
    _close_writer(writer_state)
    log(f'DONE {time.time()-t0:6.1f}s  {stats["rows"]:,} rows -> {len(writer_state["paths"])} part file(s), base {parquet_path}', label=label)
    return {'label': label, 'base_path': parquet_path, 'files': writer_state['paths'], 'period_scoped': True, **stats}


def run_month_batched(label, sql_template, columns, parquet_path, target_months,
                        numeric_cols=(), date_cols=(), stamp_col=None, rename_map=None):
    """Batches by one calendar month at a time. Used for queries parameterized by a
    single {{target_month}} rather than a start/end range. `rename_map`, if given, is
    applied to each raw chunk (e.g. quoted title-case source aliases -> snake_case)
    before the generic normalize/reindex step. Output may be split across multiple part
    files if it exceeds MAX_FILE_MB - see part_path()/PARQUET_FILES's docstring in
    CONFIG."""
    log('starting (month batched)', label=label)
    _clear_existing_parts(parquet_path)
    schema = build_schema(columns, numeric_cols, date_cols)
    log(f'target months: {target_months}', label=label)

    writer_state = _new_writer_state(parquet_path)
    stats = {'rows': 0, 'nulls': {}}
    t0 = time.time()
    for i, month in enumerate(target_months, 1):
        q = render(sql_template, target_month=month)
        month_rows = 0
        stamp = {stamp_col: pd.to_datetime(month + '-01')} if stamp_col else None
        for chunk in _iter_chunks(q):
            if len(chunk) == 0:
                continue
            if rename_map:
                chunk = chunk.rename(columns=rename_map)
            df = normalize_chunk(chunk, columns, numeric_cols, date_cols, stamp=stamp)
            _write_and_track(df, schema, writer_state, stats)
            month_rows += len(df)
        log(f'month {i}/{len(target_months)}  {month}: {month_rows:>8,} rows  (cum {stats["rows"]:>10,}, {len(writer_state["paths"])} part(s) so far)', label=label)
    _close_writer(writer_state)
    log(f'DONE {time.time()-t0:6.1f}s  {stats["rows"]:,} rows -> {len(writer_state["paths"])} part file(s), base {parquet_path}', label=label)
    return {'label': label, 'base_path': parquet_path, 'files': writer_state['paths'], 'period_scoped': True, **stats}




def run_single(label, sql_template, columns, parquet_path, numeric_cols=(), date_cols=(),
                post_process=None):
    """Runs a query once with no batching - for small reference/lookup tables only (do
    not use this for anything that scans a transaction-level fact table). Still goes
    through the same pinned-schema + size-capped-part-writer machinery as the batched
    runners, so a reference table that unexpectedly turns out huge still gets split at
    MAX_FILE_MB rather than producing one giant file. `post_process`, if given, runs on
    the full normalized DataFrame before it's written (e.g. to add a derived column)."""
    log('starting (single query, no batching - small reference table)', label=label)
    _clear_existing_parts(parquet_path)
    df = sql(sql_template, label=label)
    df = normalize_chunk(df, columns, numeric_cols, date_cols)
    if post_process:
        df = post_process(df)
    schema = build_schema(list(df.columns), numeric_cols, date_cols)
    writer_state = _new_writer_state(parquet_path)
    stats = {'rows': 0, 'nulls': {}}
    t0 = time.time()
    _write_and_track(df, schema, writer_state, stats)
    _close_writer(writer_state)
    log(f'DONE {time.time()-t0:6.1f}s  {stats["rows"]:,} rows -> {len(writer_state["paths"])} part file(s), base {parquet_path}', label=label)
    return {'label': label, 'base_path': parquet_path, 'files': writer_state['paths'], 'period_scoped': False, **stats}


# Collects every pipeline's result dict for the final Output Summary (section 19).
# Only pipelines that actually ran (RUN_FLAGS[...] == 'Yes') get an entry here.
RUN_RESULTS = {}

## 4. Query 1 — Contracts (full contract terms)

Batched by **contract_id range** (current-snapshot; `START_DATE`/`END_DATE` don't apply).
Output: `PARQUET_FILES['contracts']`.

Controlled by `RUN_FLAGS['contracts']` (set to `'No'` in CONFIG to skip).

In [ ]:
CONTRACTS_SQL = r'''WITH mapping_agg AS (
    SELECT
        cm.contract_id,
        ARRAY_AGG(
            CAST(
                ROW(cm.mapping_type, cm.id)
                AS ROW(mapping_type VARCHAR, ref_id BIGINT)
            )
        ) FILTER (WHERE cm.id IS NOT NULL) AS mappings,
        ARRAY_AGG(cm.mapped_id)
            FILTER (WHERE cm.mapping_type = 'MANUFACTURER') AS manufacturer_ids,
        ARRAY_AGG(
            cm.mapped_value
        ) FILTER (WHERE cm.mapping_type = 'MANUFACTURER') AS manufacturer_names

    FROM clm.clm_contract_mapping cm
    WHERE cm.is_deleted = 0
    GROUP BY cm.contract_id
),
raw as (
SELECT
    c.id                               AS contract_id,
    c.state                            AS contract_state,
    c.created_at,
    c.created_by,
    c.updated_at,
    c.updated_by,

    c.source_company_id              AS company_purchaser_id,
    c.source_company_name            AS company_purchaser_name,
    c.destination_company_id                  AS company_buyer_id,
    c.destination_company_name                AS company_buyer_name,
    s.name                             AS section_name,
    qa.id                              AS question_answer_id,
    qa.question_id,
    -- qa.answer,
    json_parse(qa.answer) as answer,
    qa.remarks,
    qa.created_at                      AS qa_created_at,
    qa.created_by                      AS qa_created_by,
    ma.mappings,
    ma.manufacturer_ids,
    ma.manufacturer_names,
    c.execution_date,start_date as effective_date,
    case when c.end_date is null then 'Yes' else 'No' end as valid_until_cancelled,
c.updated_at as contract_updated_at,
    json_extract_scalar(c.meta, '$.comments.MARGIN')     AS margin_comment,
    json_extract_scalar(c.meta, '$.comments.NON_MARGIN') AS non_margin_comment
FROM clm.clm_contract c
LEFT JOIN clm.clm_contract_section cs
       ON cs.contract_id = c.id
      AND cs.is_deleted = 0
LEFT JOIN clm.clm_section s
       ON s.id = cs.section_id
      AND s.is_deleted = 0
LEFT JOIN clm.clm_section_question_answer qa
       ON qa.contract_section_id = cs.id
      AND qa.is_deleted = 0
LEFT JOIN mapping_agg ma
       ON ma.contract_id = c.id
WHERE c.is_deleted = 0
and c.state not in ('ARCHIVED','REJECTED')
and c.contract_type = 'BLINKIT_EXTERNAL_TRADE'
and c.id BETWEEN {{id_from}} AND {{id_to}}
ORDER BY
    cs.section_id NULLS LAST,
    qa.question_id NULLS LAST,
    qa.id NULLS LAST

),

format_ as (
select contract_id,
case when json_extract_scalar(x, '$.Manufacturer.id') is not null then 
json_extract_scalar(x, '$.Manufacturer.id') else cast(manufacturer_ids[1] as varchar) end
AS manufacturer_id,
case when json_extract_scalar(x, '$.Manufacturer.name') is not null then
json_extract_scalar(x, '$.Manufacturer.name') else cast(manufacturer_names[1] as varchar) end
AS manufacturer,
-- company_purchaser_id,
-- company_purchaser_name,
company_buyer_id,
company_buyer_name,
contract_state,
created_at,
created_by,
updated_at,
updated_by,
execution_date,
effective_date,
valid_until_cancelled,
section_name,
question_id,
question_answer_id,
contract_updated_at,
margin_comment,
non_margin_comment,
json_extract_scalar(x,'$.value') as answer_values
from
raw
cross join unnest (CASE 
        WHEN json_extract(answer, '$.values') IS NOT NULL 
        THEN CAST(json_extract(answer, '$.values') AS array(json))
        ELSE ARRAY[answer]
    END 
) as t(x)
)

select contract_id,
manufacturer_id,
manufacturer,
company_buyer_id,
company_buyer_name,
contract_state,
date(execution_date) as execution_date,
date(effective_date) as effective_date,
date(contract_updated_at) as updated_at,
valid_until_cancelled,
margin_comment,
non_margin_comment,
min(answer_values) filter(where question_id='type_of_purchase_v1') as type_of_purchase,
min(answer_values) filter(where question_id='rtv_days_v1') as rtv_days,
max(answer_values) filter(where question_id='rtv_days_v1') as rtv_days_max,
min(answer_values) filter(where question_id='credit_days_blinkit_v1') as credit_days_purchaser,
min(answer_values) filter(where question_id='credit_days_brand_fund_v1') as credit_days_brand_fund,
min(answer_values) filter(where question_id='credit_days_penalties_v1') as credit_days_fee_penalty,
min(answer_values) filter(where question_id='credit_days_ads_business_v1') as credit_days_ads_business,
min(answer_values) filter(where question_id='right_to_set_off_v1') as right_to_set_off,
min(answer_values) filter(where question_id='pod_aligned_v1') as pod_aligned,
min(answer_values) filter(where question_id='damages_enable_v1') as damages_lost_provision,
min(answer_values) filter(where question_id='damage_tolerance_percent_v1') as damage_tolerance_percent,
min(answer_values) filter(where question_id='price_based_on_v1') as damage_price_based_on,
min(answer_values) filter(where question_id='damage_percent_landing_price_v1') as damage_percent_landing_price,
min(answer_values) filter(where question_id='recall_assist_enable_v1') as recall_assistance_fees,
min(answer_values) filter(where question_id='recall_charges_rs_v1') as recall_charges,
min(answer_values) filter(where question_id='fill_rate_enable_v1') as fill_rate_penalty,
min(answer_values) filter(where question_id='fill_rate_threshold_percent_v1') as fill_rate_threshold_percent,
min(answer_values) filter(where question_id='fill_rate_penalty_rs_v1') as fill_rate_penalty_percent,
min(answer_values) filter(where question_id='po_clubbing_enable_v1') as po_clubbing,
min(answer_values) filter(where question_id='po_clubbing_percent_monthly_po_v1') as po_clubbing_penalty,
min(answer_values) filter(where question_id='sp_benchmark_enable_v1') as sp_benchmarking,
min(answer_values) filter(where question_id='sp_benchmark_aligned_rm_percent_v1') as sp_benchmark_aligned_rm_percent,
min(answer_values) filter(where question_id='kam_support_enable_v1') as kam_support,
min(answer_values) filter(where question_id='kam_charges_rs_v1') as kam_charges,
min(answer_values) filter(where question_id='min_ads_spend_enable_v1') as min_ads_spend_enabled,
min(answer_values) filter(where question_id='min_ads_spend_value_percent_v1') as min_ads_spend_value_percent,
min(answer_values) filter(where question_id='min_ads_basis_v1') as min_ads_basis,
min(answer_values) filter(where question_id='pmc_method_v1') as purchase_margin_computation,
min(answer_values) filter(where question_id='advance_payment_enable_v1') as advance_payment,
min(answer_values) filter(where question_id='complaints_penalty_festive_v1') as complaints_penalty_festive,
min(answer_values) filter(where question_id='defective_product_penalty_v1') as defective_product_penalty,
min(answer_values) filter(where question_id='delivery_timeline_default_festive_v1') as delivery_timeline_default_festive,
min(answer_values) filter(where question_id='early_pay_enable_v1') as early_pay_enable,
min(answer_values) filter(where question_id='pm_off_invoice_enable_v1') as purchase_margin_off_invoice_enable,
min(answer_values) filter(where question_id='quality_check_penalty_festive_enable_v1') as quality_check_penalty_festive_enable,
min(answer_values) filter(where question_id='quality_check_penalty_festive_percent_v1') as quality_check_penalty_festive_percent,
min(answer_values) filter(where question_id='complaints_penalty_enable_v1') as complaints_penalty_enable,
min(answer_values) filter(where question_id='ullage_enable_v1') as ullage_enable,
min(answer_values) filter(where question_id='ullage_threshold_percent_v1') as ullage_threshold_percent,
min(answer_values) filter(where question_id='tbi_enable_v1') as target_based_incentives,
min(answer_values) filter(where question_id='tbi_payout_freq_v1') as payout_frequency

from format_
group by 1,2,3,4,5,6,7,8,9,10,11,12
'''

print(CONTRACTS_SQL[:300], '...')

In [ ]:
CONTRACTS_COLUMNS = [
    'contract_id', 'manufacturer_id', 'manufacturer', 'company_buyer_id', 'company_buyer_name',
    'contract_state', 'execution_date', 'effective_date', 'updated_at', 'valid_until_cancelled',
    'margin_comment', 'non_margin_comment', 'type_of_purchase', 'rtv_days', 'rtv_days_max',
    'credit_days_purchaser', 'credit_days_brand_fund', 'credit_days_fee_penalty',
    'credit_days_ads_business', 'right_to_set_off', 'pod_aligned', 'damages_lost_provision',
    'damage_tolerance_percent', 'damage_price_based_on', 'damage_percent_landing_price',
    'recall_assistance_fees', 'recall_charges', 'fill_rate_penalty', 'fill_rate_threshold_percent',
    'fill_rate_penalty_percent', 'po_clubbing', 'po_clubbing_penalty', 'sp_benchmarking',
    'sp_benchmark_aligned_rm_percent', 'kam_support', 'kam_charges', 'min_ads_spend_enabled',
    'min_ads_spend_value_percent', 'min_ads_basis', 'purchase_margin_computation',
    'advance_payment', 'complaints_penalty_festive', 'defective_product_penalty',
    'delivery_timeline_default_festive', 'early_pay_enable', 'purchase_margin_off_invoice_enable',
    'quality_check_penalty_festive_enable', 'quality_check_penalty_festive_percent',
    'complaints_penalty_enable', 'ullage_enable', 'ullage_threshold_percent',
    'target_based_incentives', 'payout_frequency',
]
CONTRACTS_BOUNDS_SQL = '''
SELECT MIN(id) AS min_id, MAX(id) AS max_id, COUNT(*) AS n_contracts
FROM clm.clm_contract
WHERE is_deleted = 0
  AND state NOT IN ('ARCHIVED','REJECTED')
  AND contract_type = 'BLINKIT_EXTERNAL_TRADE'
'''

if should_run('contracts'):
    RUN_RESULTS['contracts'] = run_id_batched(
        label='contracts', sql_template=CONTRACTS_SQL, columns=CONTRACTS_COLUMNS,
        parquet_path=PARQUET_FILES['contracts'], bounds_sql=CONTRACTS_BOUNDS_SQL,
        batch_size=BATCH_SIZE_CONTRACTS,
        date_cols=['execution_date', 'effective_date', 'updated_at'],
    )
else:
    log('SKIPPED (RUN_FLAGS set to No)', label='contracts')


## 5. Query 2 — Satellite City List (NEW)

**Replaces old 'satellite_fees' query.** Returns only satellite cities (MIN distance > 100km).

Batched by **date window** (period-scoped). Output: `PARQUET_FILES['satellite_city_list']`.

Controlled by `RUN_FLAGS['satellite_city_list']`.


In [ ]:
SATELLITE_CITY_LIST_SQL = r'''WITH Active_assortment AS (
    SELECT
        a.facility_id
    FROM lake_rpc.product_facility_master_assortment a
    WHERE a.active = 1
      AND master_assortment_substate_id IN (1)
      AND a.item_id IN (
            SELECT DISTINCT item_id
            FROM supply_etls.item_details
            WHERE assortment_type = 'Packaged Goods'
              AND handling_type_id <> 8
              AND manufacturer_id <> 1015
      )
    GROUP BY 1
),
ds_be_mapping AS (
    SELECT DISTINCT
        bo.facility_id    AS ds_facility_id,
        bo.be_facility_id AS be_facility_id
    FROM supply_etls.packaged_goods_inventory_and_availability bo
    INNER JOIN Active_assortment aa
        ON aa.facility_id = bo.facility_id
    WHERE bo.insert_ds_ist >= DATE('{{start_date}}')
      AND bo.insert_ds_ist < DATE('{{end_date}}')
),
distance_calc AS (
    SELECT
        ds_od.city_name,
        111 * (
            ABS(CAST(d.wh_lat AS DOUBLE) - CAST(d.ds_lat AS DOUBLE)) +
            ABS(CAST(d.wh_lon AS DOUBLE) - CAST(d.ds_lon AS DOUBLE))
        ) AS taxi_distance_km
    FROM ds_be_mapping m
    JOIN supply_etls.wh_to_ds_distance_v4 d
        ON d.ds_facility_id = m.ds_facility_id
       AND d.wh_facility_id = m.be_facility_id
    JOIN supply_etls.outlet_details ds_od
        ON ds_od.facility_id = m.ds_facility_id
)
SELECT
    city_name AS city,
    ROUND(MIN(taxi_distance_km), 4) AS min_distance_km,
    CAST('{{start_date}}' AS DATE) AS period_start,
    CAST('{{end_date}}' AS DATE) AS period_end
FROM distance_calc
GROUP BY 1
HAVING MIN(taxi_distance_km) > {{satellite_threshold}}
ORDER BY 1
'''




In [ ]:
SATCITYLIST_COLUMNS = ['city', 'min_distance_km', 'period_start', 'period_end']

if should_run('satellite_city_list'):
    RUN_RESULTS['satellite_city_list'] = run_date_batched(
        label='satellite_city_list', sql_template=SATELLITE_CITY_LIST_SQL,
        columns=SATCITYLIST_COLUMNS, parquet_path=PARQUET_FILES['satellite_city_list'],
        batch_days=BATCH_DAYS_SATCITYLIST,
        numeric_cols=['min_distance_km'],
        date_cols=['period_start', 'period_end'],
        extra_params={'satellite_threshold': SATELLITE_DISTANCE_KM_THRESHOLD},
    )
else:
    log('SKIPPED (RUN_FLAGS set to No)', label='satellite_city_list')





## 6. Query 3 — IOCC Satellite City Fee (actuals vs. sales) — UPDATED

**UPDATED:** Now filters to only satellite cities using upstream CTEs.

Batched by **date window** — period-scoped. Output: `PARQUET_FILES['satellite_city_fee']`.

**Key changes from previous version:**
- Added `Active_assortment`, `ds_be_mapping`, and `satellite_cities` CTEs
- `satellite_cities` CTE returns only cities where MIN(distance) > {{satellite_threshold}}
- INNER JOIN to `satellite_cities` filters sales to satellite cities only
- Eliminates non-satellite city records from output

Controlled by `RUN_FLAGS['satellite_city_fee']`.


In [ ]:
IOCC_SATELLITE_CITY_FEE_SQL = r'''with 
mapping_agg AS (
  SELECT
    cm.contract_id,
    ARRAY_AGG(
      CAST(
        ROW(cm.mapping_type, cm.id) AS ROW(mapping_type VARCHAR, ref_id BIGINT)
      )
    ) FILTER (
      WHERE
        cm.id IS NOT NULL
    ) AS mappings,
    ARRAY_AGG(cm.mapped_id) FILTER (
      WHERE
        cm.mapping_type = 'MANUFACTURER'
    ) AS manufacturer_ids,
    ARRAY_AGG(cm.mapped_value) FILTER (
      WHERE
        cm.mapping_type = 'MANUFACTURER'
    ) AS manufacturer_names
  
  FROM
    clm.clm_contract_mapping cm
  WHERE
    cm.is_deleted = 0 -- and cm.contract_id=138
  GROUP BY
    cm.contract_id
),

Active_assortment AS (
    SELECT
        a.facility_id
    FROM lake_rpc.product_facility_master_assortment a
    WHERE a.active = 1
      AND master_assortment_substate_id IN (1)
      AND a.item_id IN (
            SELECT DISTINCT item_id
            FROM supply_etls.item_details
            WHERE assortment_type = 'Packaged Goods'
              AND handling_type_id <> 8
              AND manufacturer_id <> 1015
      )
    GROUP BY 1
),
ds_be_mapping AS (
    SELECT DISTINCT
        bo.facility_id    AS ds_facility_id,
        bo.be_facility_id AS be_facility_id
    FROM supply_etls.packaged_goods_inventory_and_availability bo
    INNER JOIN Active_assortment aa ON aa.facility_id = bo.facility_id
),
satellite_cities AS (
    SELECT DISTINCT city_name
    FROM ds_be_mapping m
    JOIN supply_etls.wh_to_ds_distance_v4 d
        ON d.ds_facility_id = m.ds_facility_id
       AND d.wh_facility_id = m.be_facility_id
    JOIN supply_etls.outlet_details ds_od
        ON ds_od.facility_id = m.ds_facility_id
    WHERE 111 * (
            ABS(CAST(d.wh_lat AS DOUBLE) - CAST(d.ds_lat AS DOUBLE)) +
            ABS(CAST(d.wh_lon AS DOUBLE) - CAST(d.ds_lon AS DOUBLE))
        ) > {{satellite_threshold}}
    AND ds_od.city_name IS NOT NULL
),

raw as (
  SELECT
    c.id AS contract_id,
    c.state AS contract_state,
    c.created_at,
    c.created_by,
    c.updated_at,
    c.updated_by,
    c.source_company_id AS company_purchaser_id,
    c.source_company_name AS company_purchaser_name,
    c.destination_company_id AS company_buyer_id,
    c.destination_company_name AS company_buyer_name,
    s.name AS section_name,
    qa.id AS question_answer_id,
    qa.question_id,
    -- qa.answer,
    json_parse(qa.answer) as answer,
    qa.remarks,
    qa.created_at AS qa_created_at,
    qa.created_by AS qa_created_by,
    ma.mappings,
    ma.manufacturer_ids,
    ma.manufacturer_names
  FROM
    clm.clm_contract c
    LEFT JOIN clm.clm_contract_section cs ON cs.contract_id = c.id
    AND cs.is_deleted = 0
    JOIN clm.clm_section s ON s.id = cs.section_id
    AND s.is_deleted = 0
    and s.name = 'Satellite Location Fees'
    LEFT JOIN clm.clm_section_question_answer qa ON qa.contract_section_id = cs.id
    AND qa.is_deleted = 0
    LEFT JOIN mapping_agg ma ON ma.contract_id = c.id
  WHERE
    c.is_deleted = 0
  ORDER BY
    cs.section_id NULLS LAST,
    qa.question_id NULLS LAST,
    qa.id NULLS LAST
),

format_ as (
  select
    contract_id,
    case
      when json_extract_scalar(x, '$.Manufacturer.id') is not null then json_extract_scalar(x, '$.Manufacturer.id')
      else cast(manufacturer_ids [1] as varchar)
    end AS manufacturer_id,
    case
      when json_extract_scalar(x, '$.Manufacturer.name') is not null then json_extract_scalar(x, '$.Manufacturer.name')
      else cast(manufacturer_names [1] as varchar)
    end AS manufacturer,
    contract_state,
    created_at,
    created_by,
    updated_at,
    updated_by,
    company_purchaser_id,
    company_purchaser_name,
    company_buyer_id,
    company_buyer_name,
    section_name,
    question_answer_id,
    question_id,
    json_extract_scalar(x, '$.value') as answer_values,
    x,
    manufacturer_ids,
    manufacturer_names
  from
    raw
    cross join unnest (
      CASE
        WHEN json_extract(answer, '$.values') IS NOT NULL THEN CAST(json_extract(answer, '$.values') AS array(json))
        ELSE ARRAY [answer]
      END
    ) as t(x)
),

details as (
  select
    contract_id,
    json_extract_scalar(row, '$.product_type') AS product_type,
    json_extract_scalar(row, '$.bucket') AS bucket,
    CAST(json_extract_scalar(row, '$.item_mrp') AS double) AS item_mrp,
    json_extract_scalar(row, '$.charge_type') AS charge_type,
    CAST(json_extract_scalar(row, '$.value') AS double) AS fee_amt,
    json_extract_scalar(row, '$.max_fee_threshold') AS max_fee_threshold,
    json_extract_scalar(row, '$.max_fee_amt') AS max_fee_amt
  from
    format_
    cross join unnest(
      CAST(json_extract(x, '$.value.rows') AS array(json))
    ) AS t(row)
  where
    question_id = 'satellite_fees_table_v1'
)

,contract_dets as(
select
  f.manufacturer_id,
  manufacturer,
  min(answer_values) filter(
    where
      question_id = 'satellite_fees_enable_v1'
  ) as satellite_fee,
  case when product_type = 'Cold' then 'Y' else 'N' end as is_cold,
  coalesce(bucket, '0') bucket,
  coalesce(item_mrp, 0.0) as item_mrp,
  coalesce(fee_amt, 0.0) as fee_amt,
  coalesce(max_fee_threshold, 'No') as max_fee_threshold,
  coalesce(max_fee_amt, '0') as max_fee_amt

from format_ f
left join details d on f.contract_id = d.contract_id
-- where f.manufacturer_id = '40'

group by 1,2,4,5,6,7,8,9
  )

,sor as(
with tagg as (
with manf_details as (
    SELECT 
        pp.item_id,
        pp.name as item_name,
        pp.brand_id,
        pm.id as manufacturer_id,
        pm.name as manufacturer,
        b.name as brand,
        pp.shelf_life
    FROM rpc.product_product pp 
        INNER JOIN (SELECT item_id, max(id) as max_id
                    FROM rpc.product_product 
                    WHERE approved=1 and active = 1 and lake_active_record = true
                    GROUP BY 1) a ON a.item_id = pp.item_id AND a.max_id = pp.id
        INNER JOIN rpc.product_brand b on b.id = pp.brand_id and b.active = 1 AND b.lake_active_record = true
        INNER JOIN rpc.product_manufacturer pm on pm.id = b.manufacturer_id and pm.active = 1 AND pm.lake_active_record = true

    GROUP BY 1,2,3,4,5,6,7
    )
    
select
    coalesce(m.item_id,mm.item_id,mmm.item_id) as item_id,
    0 as flag,
    r.return_type,
    max(case when rr.bucket_name=1 then 1 else 0 end) as good,
    max(case when rr.bucket_name=2 then 1 else 0 end) as damage,
    max(case when rr.bucket_name=3 then 1 else 0 end) as NTE

from rpc.return_policy r
inner join rpc.return_policy_bucket_attribute rr on r.id=rr.policy_id
left join manf_details m on m.manufacturer_id=r.reference_id and r.scope_type=3
left join manf_details mm on mm.brand_id=r.reference_id and r.scope_type=2
left join manf_details mmm on mmm.item_id=r.reference_id

where 
    r.active = true
    and rr.active = true
    and r.scope_type = 1
    and r.return_type = 2
group by 1,2,3

UNION 

select
    coalesce(m.item_id,mm.item_id,mmm.item_id) as item_id,
    1 as flag,
    r.return_type,
    max(case when rr.bucket_name=1 then 1 else 0 end) as good,
    max(case when rr.bucket_name=2 and r.reference_id not in 
    (318,326,514,516,533,628,774,807,1101,1418,1697,1712,1772,1798,1834,1847,1918,1944,1959,1973,2068,2197,2225,2310,2552,2790,2798,2809,2823,2862,2872,2879,2881,2891,2936,2937,2958,2959,2971,2972,3007,3024,3168,3169,3178,3199,3268,3269,3279,3310,3344,3350,3356,3407,3408,3443,3460,3462,3489,3532,3565,3589,3750,3786,3806,3821,3887,3981,4349,4359,4414,4446,4522,4556,4573,4620,4649)
    then 1 else 0 end) as damage,
    max(case when rr.bucket_name=3 then 1 else 0 end) as NTE

from rpc.return_policy r
inner join rpc.return_policy_bucket_attribute rr on r.id=rr.policy_id
left join manf_details m on m.manufacturer_id=r.reference_id and r.scope_type=3
left join manf_details mm on mm.brand_id=r.reference_id and r.scope_type=2
left join manf_details mmm on mmm.item_id=r.reference_id

where 
    r.active = true
    and r.scope_type = 3
    and rr.active = true
    and r.return_type = 2
group by 1,2,3
)

select tagg.item_id

from tagg
join (select item_id, min(flag) as flag from tagg group by 1) f on tagg.item_id = f.item_id and tagg.flag = f.flag
group by 1
)

,wh_to_ds_dist as(
select 
    ds_outlet_id, 
    wh_outlet_id,
    distance_km,
    row_number() over (partition by ds_outlet_id order by distance_km) as dist_rank

from supply_etls.wh_to_ds_distance_v4 a
join category_etls.tea_tagging_category b on cast(b.be_outlet_id as integer) = a.wh_outlet_id 
        and cast(b.fe_outlet_id as integer) = a.ds_outlet_id
order by 1,3
)

,wh_with_min_dist as(
select *, case when distance_km > 100 then 1 else 0 end as satellite_flag from wh_to_ds_dist where dist_rank = 1
)

,map as (
select *,count(1) over (partition by product_id) cnt 
from
    (select
    om.product_id,id.item_id,id.name,p.product_name, id.storage_type,
    (om.avg_selling_price_ratio) * case when avg_selling_price_ratio > 0 then cast(1 as double)/cast(sum(om.avg_selling_price_ratio) over (partition by om.product_id) as double) else 0 end as avg_selling_price_ratio,
    (om.avg_mrp_ratio) * case when avg_mrp_ratio > 0 then cast(1 as double)/cast(sum(om.avg_mrp_ratio) over (partition by om.product_id) as double) else 0 end as avg_mrp_ratio,
    (multiplier) multiplier
    
    from dwh.dim_item_product_offer_mapping om
    left join lake_rpc.item_details id on om.item_id=id.item_id
    left join dwh.dim_product p on p.product_id=om.product_Id and p.is_current
    where om.is_current
    )
)
select * from (
select
    {{day_week_month}} as period,
    city,
    distance_km,
    item_id,
    sor_item_id,
    item_name, 
    x.manufacturer_id,
    x.manufacturer,
    x.is_cold,
    sor_flag,
    unit_mrp,
    sale_qty,
    case when
                bucket = 'Item MRP ≥ X' and unit_mrp >= item_mrp then 'Yes'
             when
                bucket = 'Item MRP < X' and unit_mrp < item_mrp then 'Yes'
             else 'No'
             end as bucket,
    
    (case when
                bucket = 'Item MRP ≥ X' and unit_mrp >= item_mrp then (sale_qty)*fee_amt
             when
                bucket = 'Item MRP < X' and unit_mrp < item_mrp then (sale_qty)*fee_amt
             end
    ) as satellite_fee

from (
select
    date(date_trunc('{{day_week_month}}',fs.order_deliver_ts_ist)) as {{day_week_month}},
    city,
    distance_km,
    cast(m.item_id as varchar) as item_id,
    sor.item_id as sor_item_id,
    item_name, 
    manufacturer_id,
    manufacturer,
    case when storage_type = '11' then 'Y' else 'N' end as is_cold,
    case when sor.item_id is not null then 'sor' else 'outright' end as sor_flag,
    (oi.unit_mrp*cast(coalesce(m.avg_mrp_ratio, 1.0) as double)) as unit_mrp,
    
    sum(oi.procured_quantity*coalesce(m.multiplier,1.0)) as sale_qty,
    sum(oi.unit_mrp*oi.procured_quantity*cast(coalesce(m.avg_mrp_ratio, 1.0) as double)) as sale_mrp_amt

from dwh.fact_sales_order_item_details oi
left join dwh.fact_sales_order_details fs on oi.order_id=fs.order_id
left join dwh.dim_product c on c.product_id = oi.product_id and c.is_current
left join map m on oi.product_id = m.product_id
left join (select outlet_id, max(city) city, max(outlet_name) outlet_name from ba_etls.outlet_details group by 1) o on oi.outlet_id = o.outlet_id
        inner join satellite_cities sc on o.city = sc.city_name
left join wh_with_min_dist ds on ds.ds_outlet_id = oi.outlet_id
left join (select item_id, max(item_name) as item_name, max(manufacturer) manufacturer
                , cast(max(manufacturer_id) as varchar) manufacturer_id from ba_etls.item_details group by 1) i on m.item_id = i.item_id
left join sor on sor.item_id = m.item_id  

where oi.order_create_dt_ist >= date'{{start_date}}' - interval '1' day
    and oi.order_create_dt_ist < date'{{end_date}}'  + interval '1' day
    and fs.order_create_dt_ist >=  date'{{start_date}}'  - interval '1' day
    and fs.order_deliver_ts_ist >=  date'{{start_date}}' 
    and fs.order_deliver_ts_ist <  date'{{end_date}}'  + interval '1' day
    and oi.is_internal_order = false
    and oi.procured_quantity > 0
    and oi.unit_selling_price > 0
    and oi.order_current_status = 'DELIVERED'
    and coalesce(c.brand_name,'blank') != 'Apple'
    and satellite_flag = 1
    and case when '{{Mfr ID}}' = '%%' then manufacturer_id <> '' else manufacturer_id = '{{Mfr ID}}' end
    and case when '{{Mfr Name}}' = '%%' then manufacturer <> '' else lower(manufacturer) like lower('%%{{Mfr Name}}%%') end
    -- and city = 'Rajkot'
    -- and m.item_id = 10002915

group by 1,2,3,4,5,6,7,8,9,10,11
)x
left join contract_dets cd on cd.manufacturer_id = x.manufacturer_id and cd.is_cold = x.is_cold and cd.satellite_fee = 'Yes'
-- where cd.manufacturer_id = '40'
)
where bucket = 'Yes'
'''

print(IOCC_SATELLITE_CITY_FEE_SQL[:300], '...')

In [ ]:
SATCITYFEE_COLUMNS = [
    'period', 'city', 'distance_km', 'item_id', 'sor_item_id', 'item_name',
    'manufacturer_id', 'manufacturer', 'is_cold', 'sor_flag', 'unit_mrp', 'sale_qty',
    'bucket', 'satellite_fee',
]

if should_run('satellite_city_fee'):
    RUN_RESULTS['satellite_city_fee'] = run_date_batched(
        label='satellite_city_fee', sql_template=IOCC_SATELLITE_CITY_FEE_SQL, columns=SATCITYFEE_COLUMNS,
        parquet_path=PARQUET_FILES['satellite_city_fee'], batch_days=BATCH_DAYS_SATCITYFEE,
        numeric_cols=['distance_km', 'unit_mrp', 'sale_qty', 'satellite_fee'],
        date_cols=['period'],
        extra_params={'day_week_month': DAY_WEEK_MONTH, 'Mfr ID': MFR_ID, 'Mfr Name': MFR_NAME, 'satellite_threshold': SATELLITE_DISTANCE_KM_THRESHOLD},
    )
else:
    log('SKIPPED (RUN_FLAGS set to No)', label='satellite_city_fee')


## 7. Query 4 — IOCC KAM Fees as aligned in Contracts

Batched by **contract_id range** (current-snapshot). Output: `PARQUET_FILES['kam_fees']`.

Controlled by `RUN_FLAGS['kam_fees']` (set to `'No'` in CONFIG to skip).

In [ ]:
KAM_FEES_SQL = r'''WITH mapping_agg AS (
    SELECT
        cm.contract_id,
        ARRAY_AGG(
            CAST(
                ROW(cm.mapping_type, cm.id)
                AS ROW(mapping_type VARCHAR, ref_id BIGINT)
            )
        ) FILTER (WHERE cm.id IS NOT NULL) AS mappings,
        ARRAY_AGG(cm.mapped_id)
            FILTER (WHERE cm.mapping_type = 'MANUFACTURER') AS manufacturer_ids,
        ARRAY_AGG(
            cm.mapped_value
        ) FILTER (WHERE cm.mapping_type = 'MANUFACTURER') AS manufacturer_names
    FROM clm.clm_contract_mapping cm
    WHERE cm.is_deleted = 0
    GROUP BY cm.contract_id
),
raw as (
SELECT
    c.id                               AS contract_id,
    c.state                            AS contract_state,
    c.created_at,
    c.created_by,
    c.updated_at,
    c.updated_by,
    c.source_company_id              AS company_purchaser_id,
    c.source_company_name            AS company_purchaser_name,
    c.destination_company_id                  AS company_buyer_id,
    c.destination_company_name                AS company_buyer_name,
    s.name                             AS section_name,
    qa.id                              AS question_answer_id,
    qa.question_id,
    -- qa.answer,
    json_parse(qa.answer) as answer,
    qa.remarks,
    qa.created_at                      AS qa_created_at,
    qa.created_by                      AS qa_created_by,
    ma.mappings,
    ma.manufacturer_ids,
    ma.manufacturer_names,
    c.execution_date,start_date as effective_date,
    case when c.end_date is null then 'Yes' else 'No' end as valid_until_cancelled,
c.updated_at as contract_updated_at
FROM clm.clm_contract c
LEFT JOIN clm.clm_contract_section cs
       ON cs.contract_id = c.id
      AND cs.is_deleted = 0
LEFT JOIN clm.clm_section s
       ON s.id = cs.section_id
      AND s.is_deleted = 0
LEFT JOIN clm.clm_section_question_answer qa
       ON qa.contract_section_id = cs.id
      AND qa.is_deleted = 0
LEFT JOIN mapping_agg ma
       ON ma.contract_id = c.id
WHERE c.is_deleted = 0
and c.state not in ('ARCHIVED','REJECTED')
and c.id BETWEEN {{id_from}} AND {{id_to}}
ORDER BY
    cs.section_id NULLS LAST,
    qa.question_id NULLS LAST,
    qa.id NULLS LAST
),
format_ as (
select contract_id,
case when json_extract_scalar(x, '$.Manufacturer.id') is not null then 
json_extract_scalar(x, '$.Manufacturer.id') else cast(manufacturer_ids[1] as varchar) end
AS manufacturer_id,
case when json_extract_scalar(x, '$.Manufacturer.name') is not null then
json_extract_scalar(x, '$.Manufacturer.name') else cast(manufacturer_names[1] as varchar) end
AS manufacturer,
-- company_buyer_id,
-- company_buyer_name,
company_buyer_id,
company_buyer_name,
contract_state,
created_at,
created_by,
updated_at,
updated_by,
execution_date,
effective_date,
valid_until_cancelled,
section_name,
question_id,
question_answer_id,
contract_updated_at,
json_extract_scalar(x,'$.value') as answer_values
from
raw
cross join unnest (CASE 
        WHEN json_extract(answer, '$.values') IS NOT NULL 
        THEN CAST(json_extract(answer, '$.values') AS array(json))
        ELSE ARRAY[answer]
    END 
) as t(x)
),
fff as (
select contract_id,
manufacturer_id,
manufacturer,
-- company_buyer_id,
-- company_buyer_name,
contract_state,
date(execution_date) as execution_date,
date(effective_date) as effective_date,
-- date(contract_updated_at) as updated_at,/
valid_until_cancelled,
min(answer_values) filter(where question_id='kam_support_enable_v1') as kam_support,
min(answer_values) filter(where question_id='kam_charges_rs_v1') as kam_charges
from format_
group by 1,2,3,4,5,6,7--,8,9
)
select * from fff where kam_support='Yes'
'''

print(KAM_FEES_SQL[:300], '...')

In [ ]:
KAM_COLUMNS = [
    'contract_id', 'manufacturer_id', 'manufacturer', 'contract_state', 'execution_date',
    'effective_date', 'valid_until_cancelled', 'kam_support', 'kam_charges',
]
KAM_BOUNDS_SQL = '''
SELECT MIN(id) AS min_id, MAX(id) AS max_id, COUNT(*) AS n_contracts
FROM clm.clm_contract
WHERE is_deleted = 0
  AND state NOT IN ('ARCHIVED','REJECTED')
'''

if should_run('kam_fees'):
    RUN_RESULTS['kam_fees'] = run_id_batched(
        label='kam_fees', sql_template=KAM_FEES_SQL, columns=KAM_COLUMNS,
        parquet_path=PARQUET_FILES['kam_fees'], bounds_sql=KAM_BOUNDS_SQL,
        batch_size=BATCH_SIZE_KAM,
        date_cols=['execution_date', 'effective_date'],
    )
else:
    log('SKIPPED (RUN_FLAGS set to No)', label='kam_fees')


## 8. Query 5 — BCPL Defective CRBS

Batched by **date window** — period-scoped, hits complaints/returns/sales-invoice fact
tables. Output: `PARQUET_FILES['bcpl_defective_crbs']`. Penalty is a flat ₹50/unit (hardcoded
in the source query, not from a contract terms table).

Controlled by `RUN_FLAGS['bcpl_defective_crbs']` (set to `'No'` in CONFIG to skip).

In [ ]:
BCPL_DEFECTIVE_CRBS_SQL = r'''WITH
base as(
select 
    cart_id,
    req_items.product_id,
    a.complaint_type,
    row_number() over (partition by cart_id, product_id order by complaint_type) as rn

from zomato.dynamodb.blinkit_crm_complaints a
LEFT JOIN UNNEST(req_items) AS req_items ON TRUE

where dt >= REPLACE(cast(date('{{start_date}}') - interval '15' day as varchar), '-', '')
    and dt <= REPLACE('{{end_date}}', '-', '')
    and complaint_type not in ('SYSTEM_PROCUREMENT_UPDATE','SYSTEM_PROCUREMENT_UPDATE'
    ,'SYSTEM_PAY_BEFORE_DELIVERY','SYSTEM_PAY_BEFORE_DELIVERY','SYSTEM_CANCEL',
    'SYSTEM_NOT_PROCURED','SYSTEM_PRICE_CHANGED')
    
GROUP BY 1, 2, 3
)

,base2 as(select cart_id, product_id, complaint_type from base where rn = 1 group by 1,2,3)

,base3 as(
select
    cart_id,
    suborder_id,
    i.item_id,
    p.product_id,
    source_entity_vendor_id,
    lower(source_entity_vendor_legal_name) as vendor_legal_name,
    sum(unit_forward_mrp*total_net_quantity) as gmv

from dwh.fact_sales_invoice_item_details i
left join (select item_id, product_id from dwh.dim_item_product_offer_mapping group by 1,2) p on p.item_id = i.item_id

where i.cart_checkout_date_ist >= date'{{start_date}}' - interval '30' day
    and i.cart_checkout_date_ist < date'{{end_date}}' + interval '1' day
    and i.order_create_date_ist >= date'{{start_date}}' - interval '40' day
    and cart_id in (select cart_id from base group by 1)

group by 1,2,3,4,5,6
)

,remarks as(
select order_id, product_id, customer_message from cd_etls.complaint_session_customer_comments 
where date_ between '{{start_date}}' and cast(date'{{end_date}}' + interval '30' day as varchar)
group by 1,2,3
)

,base4 as(
select
    case when '{{day_week_month}}' = 'week' then date(date_trunc('week', day)) 
        when '{{day_week_month}}' = 'day' then date(date_trunc('day', day)) 
        when '{{day_week_month}}' = 'month' then date(date_trunc('month', day)) 
        end as duration,
    v.outlet_id,
    -- co.name as outlet_name,
    -- cl.name as city_name,
    v.cart_id,
    v.order_id,
    v.product_id,
    -- p.product_name,
    item_id,
    -- CASE WHEN l0_category_id in (1487) then 'fnv'
    --         WHEN l0_category_id in (888,15,4,14,12) 
    --         and l1_category_id in (279,1425,1361,1362,1363,1388,922,123,923,1200,953,1184) 
    --         and l2_category_id in (1956,1425,63,1367,1369,1730,127,1094,949,138,1091,1185,123,1778,950,1389,31,1097,198,1093) then 'perishable'
    --         ELSE 'packaged' end as category,
    -- l0_category,
    -- l1_category,
    -- l2_category,
    -- product_type,
    -- brand_name,
    -- manufacturer_name,
    -- unit,
    b.source_entity_vendor_id as entity_id,
    b.vendor_legal_name as entity_name,
    complaint_type,
    remarks,
    case when complaint_type in ('COMPLAINT_AUDIO_ISSUE','COMPLAINT_BLUETOOTH_ISSUE','COMPLAINT_COLOR_ISSUE', 'COMPLAINT_DAMAGED_ITEM',
        'COMPLAINT_EXPECTATION_MISMATCH', 'COMPLAINT_FAKE_OR_DUPLICATE_ITEM', 'COMPLAINT_FAULTY_ITEM', 'COMPLAINT_FREEBIE_MISSING',
        'COMPLAINT_MISSING_PART', 'COMPLAINT_PACKAGING_ISSUE','COMPLAINT_PAPER_QUALITY_ISSUE','COMPLAINT_PARTIAL_ITEM_MISSING',
        'COMPLAINT_PRICE_MISMATCH','COMPLAINT_PRINT_QUALITY_ISSUE','COMPLAINT_QUALITY_ISSUE','COMPLAINT_SMELL_OR_TASTE_ISSUE',
        'COMPLAINT_SIZE_ISSUE') then 'defective' else 'others' end 
        as defective,
    
    
    count(distinct v.cart_id) as order_count,
    sum(coalesce(bad_returns,0.0)) bad_returns,
    sum(coalesce(bad_returns_qty,0)) bad_returns_qty,
    sum(gmv) as gmv

from dwh.flat_order_item_returns v
left join base3 b on b.cart_id = v.cart_id and b.suborder_id = v.suborder_id and b.product_id = v.product_id
left join base2 b2 on b2.product_id = v.product_id and b2.cart_id = v.cart_id
left join (select order_id, product_id, listagg(customer_message, ', ') within group (order by order_id) as remarks from remarks group by 1,2) r on r.order_id = v.order_id and r.product_id = v.product_id
-- left join retail.console_outlet co ON co.id = v.outlet_id
-- left join retail.console_location cl on cl.id = co.tax_location_id 
-- left join dwh.dim_product p on v.product_id = p.product_id and p.is_product_enabled = true and p.is_current = true

where day between date'{{start_date}}' and date'{{end_date}}'
    and v.product_id <> 0
    and bad_returns_qty <> 0
    and complaint_type in ('COMPLAINT_AUDIO_ISSUE','COMPLAINT_BLUETOOTH_ISSUE','COMPLAINT_COLOR_ISSUE', 'COMPLAINT_DAMAGED_ITEM',
        'COMPLAINT_EXPECTATION_MISMATCH', 'COMPLAINT_FAKE_OR_DUPLICATE_ITEM', 'COMPLAINT_FAULTY_ITEM', 'COMPLAINT_FREEBIE_MISSING',
        'COMPLAINT_MISSING_PART', 'COMPLAINT_PACKAGING_ISSUE','COMPLAINT_PAPER_QUALITY_ISSUE','COMPLAINT_PARTIAL_ITEM_MISSING',
        'COMPLAINT_PRICE_MISMATCH','COMPLAINT_PRINT_QUALITY_ISSUE','COMPLAINT_QUALITY_ISSUE','COMPLAINT_SMELL_OR_TASTE_ISSUE',
        'COMPLAINT_SIZE_ISSUE')
    -- and v.order_id = 1136161923

group by 1,2,3,4,5,6,7,8,9,10,11--,12--,13--,14,15,16,17,18,19,20,21
)

select
    duration,
    -- cast(b.order_id as varchar) order_id,
    cast(entity_id as varchar) entity_id,
    entity_name,
    -- cast(b.product_id as varchar) product_id,
    cast(b.item_id as varchar) item_id,
    item_name, 
    cast(manufacturer_id as varchar) manufacturer_id,
    manufacturer,
    complaint_type,
    remarks,
    -- product_name,
    
    sum(bad_returns_qty) as bad_returns_qty,
    sum(bad_returns_qty)*50 as penalty,
    sum(gmv) as gmv,
    sum(bad_returns_qty)*50.00/sum(gmv) as pct_gmv

from base4 b
left join (select item_id, max(item_name) as item_name, max(manufacturer) manufacturer
                , cast(max(manufacturer_id) as varchar) manufacturer_id from ba_etls.item_details group by 1) i on b.item_id = i.item_id
where defective = 'defective'
    and case when '{{Mfr ID}}' = '%%' then manufacturer_id <> '' else manufacturer_id = '{{Mfr ID}}' end
    and case when '{{Mfr Name}}' = '%%' then manufacturer <> '' else lower(manufacturer) like lower('%%{{Mfr Name}}%%') end
    -- manufacturer = '{{Mfr Name}}'

group by 1,2,3,4,5,6,7,8,9--,10,11
'''

print(BCPL_DEFECTIVE_CRBS_SQL[:300], '...')

In [ ]:
BCPL_COLUMNS = [
    'duration', 'entity_id', 'entity_name', 'item_id', 'item_name', 'manufacturer_id',
    'manufacturer', 'complaint_type', 'remarks', 'bad_returns_qty', 'penalty', 'gmv', 'pct_gmv',
]

if should_run('bcpl_defective_crbs'):
    RUN_RESULTS['bcpl_defective_crbs'] = run_date_batched(
        label='bcpl_defective_crbs', sql_template=BCPL_DEFECTIVE_CRBS_SQL, columns=BCPL_COLUMNS,
        parquet_path=PARQUET_FILES['bcpl_defective_crbs'], batch_days=BATCH_DAYS_BCPL,
        numeric_cols=['bad_returns_qty', 'penalty', 'gmv', 'pct_gmv'],
        date_cols=['duration'],
        extra_params={'day_week_month': DAY_WEEK_MONTH, 'Mfr ID': MFR_ID, 'Mfr Name': MFR_NAME},
    )
else:
    log('SKIPPED (RUN_FLAGS set to No)', label='bcpl_defective_crbs')


## 9. Query 6 — IOCC Recall Assistance (PRN-based)

Batched by **date window** — period-scoped, hits POS invoice tables. Output:
`PARQUET_FILES['recall_assistance']`. Source query's trailing `LIMIT 100` (with no
`ORDER BY`) was removed — left in, it would have silently capped the whole result set.

Controlled by `RUN_FLAGS['recall_assistance']` (set to `'No'` in CONFIG to skip).

In [ ]:
RECALL_ASSISTANCE_PRN_SQL = r'''WITH variants AS (
  SELECT
    DISTINCT variant_id,
    cast(manufacturer_id as varchar) as manufacturer_id
  FROM
    po.purchase_order_items
),

unique_tax_info AS (
  SELECT
    vendor_city_id,
    legal_name,
    regexp_replace(upper(trim(legal_name)), '[^a-zA-Z0-9]', '') as normalized_name,
    ROW_NUMBER() OVER(
      PARTITION BY vendor_city_id
      ORDER BY
        id DESC
    ) as rn
  FROM
    vms.vms_vendor_city_tax_info
  WHERE
    active = 1
    and lake_active_record = true
)

,mapping_agg AS (
    SELECT
        cm.contract_id,
        ARRAY_AGG(
            CAST(
                ROW(cm.mapping_type, cm.id)
                AS ROW(mapping_type VARCHAR, ref_id BIGINT)
            )
        ) FILTER (WHERE cm.id IS NOT NULL) AS mappings,
        ARRAY_AGG(cm.mapped_id)
            FILTER (WHERE cm.mapping_type = 'MANUFACTURER') AS manufacturer_ids,
        ARRAY_AGG(
            cm.mapped_value
        ) FILTER (WHERE cm.mapping_type = 'MANUFACTURER') AS manufacturer_names

    FROM clm.clm_contract_mapping cm
    WHERE cm.is_deleted = 0
    GROUP BY cm.contract_id
)
,
raw as (
SELECT
    c.id                               AS contract_id,
    c.state                            AS contract_state,
    c.created_at,
    c.created_by,
    c.updated_at,
    c.updated_by,

    c.source_company_id              AS company_purchaser_id,
    c.source_company_name            AS company_purchaser_name,
    c.destination_company_id                  AS company_buyer_id,
    c.destination_company_name                AS company_buyer_name,
    s.name                             AS section_name,
    qa.id                              AS question_answer_id,
    qa.question_id,
    -- qa.answer,
    json_parse(qa.answer) as answer,
    qa.remarks,
    qa.created_at                      AS qa_created_at,
    qa.created_by                      AS qa_created_by,
    ma.mappings,
    ma.manufacturer_ids,
    ma.manufacturer_names,
    c.execution_date,start_date as effective_date,
    case when c.end_date is null then 'Yes' else 'No' end as valid_until_cancelled,
c.updated_at as contract_updated_at
FROM clm.clm_contract c
LEFT JOIN clm.clm_contract_section cs
       ON cs.contract_id = c.id
      AND cs.is_deleted = 0
LEFT JOIN clm.clm_section s
       ON s.id = cs.section_id
      AND s.is_deleted = 0
LEFT JOIN clm.clm_section_question_answer qa
       ON qa.contract_section_id = cs.id
      AND qa.is_deleted = 0
LEFT JOIN mapping_agg ma
       ON ma.contract_id = c.id
WHERE c.is_deleted = 0
and c.state <> 'ARCHIVED'
ORDER BY
    cs.section_id NULLS LAST,
    qa.question_id NULLS LAST,
    qa.id NULLS LAST

),

format_ as (
select 
    contract_id,
    case when json_extract_scalar(x, '$.Manufacturer.id') is not null then 
        json_extract_scalar(x, '$.Manufacturer.id') else cast(manufacturer_ids[1] as varchar) end AS manufacturer_id,
    case when json_extract_scalar(x, '$.Manufacturer.name') is not null then
    json_extract_scalar(x, '$.Manufacturer.name') else cast(manufacturer_names[1] as varchar) end AS manufacturer,
    -- company_purchaser_id,
    -- company_purchaser_name,
    company_buyer_id,
    company_buyer_name,
    contract_state,
    created_at,
    created_by,
    updated_at,
    updated_by,
    execution_date,
    effective_date,
    valid_until_cancelled,
    section_name,
    question_id,
    question_answer_id,
    contract_updated_at,
    json_extract_scalar(x,'$.value') as answer_values

from raw
cross join unnest (CASE 
        WHEN json_extract(answer, '$.values') IS NOT NULL 
        THEN CAST(json_extract(answer, '$.values') AS array(json))
        ELSE ARRAY[answer]
    END 
) as t(x)
-- where contract_state <> 'ARCHIVED'
)

,recall as(
select 
    manufacturer_id,
    manufacturer, 
    recall_charges

from(
select 
    contract_id,
    manufacturer_id,
    manufacturer,
    company_buyer_id,
    company_buyer_name,
    contract_state,
    date(execution_date) as execution_date,
    date(effective_date) as effective_date,
    date(contract_updated_at) as updated_at,
    valid_until_cancelled,
    min(answer_values) filter(where question_id='recall_assist_enable_v1') as Recall_Assistance_Fees,
min(answer_values) filter(where question_id='recall_charges_rs_v1') as recall_charges

from format_
group by 1,2,3,4,5,6,7,8,9,10
)
where Recall_Assistance_Fees = 'Yes' --and contract_state <> 'ARCHIVED'
group by 1,2,3
)


  SELECT
    case when '{{day_week_month}}' = 'week' then date(date_trunc('week', p.pos_timestamp + interval '330' minute)) 
        when '{{day_week_month}}' = 'day' then date(date_trunc('day', p.pos_timestamp + interval '330' minute)) 
        when '{{day_week_month}}' = 'month' then date(date_trunc('month', p.pos_timestamp + interval '330' minute)) 
        end as duration,
    var.manufacturer_id,
    p2.manufacturer,
    item_id,
    p2.name as item_name,
    upper(trim(cti.legal_name)) AS entity,
    CAST(SUM(p1.quantity) AS BIGINT) AS prn_qty,
    CAST(SUM(p1.quantity*cast(recall_charges as double)) AS BIGINT) as recall_charges
  
  FROM
    pos.pos_invoice p
    INNER JOIN pos.pos_invoice_product_details p1 ON p.id = p1.invoice_id
        and p1.insert_ds_ist >= '{{start_date}}'
        and p1.insert_ds_ist <= cast(date '{{end_date}}' + interval '1' day as varchar)
        and p.lake_active_record = true and p1.lake_active_record = true
    INNER JOIN rpc.product_product p2 ON p2.variant_id = p1.variant_id and p2.lake_active_record = true and p2.lake_active_record = true
    LEFT JOIN variants var ON p2.variant_id = var.variant_id 
    LEFT JOIN vms.vms_vendor_city_mapping cm ON cm.vendor_id = p.source_entity_vendor_id
        AND cm.active = 1 and cm.lake_active_record = true
    LEFT JOIN unique_tax_info cti ON cti.vendor_city_id = cm.id
        AND cti.rn = 1
    left join recall rc on rc.manufacturer_id = var.manufacturer_id
  
  WHERE
    p.invoice_type_id = 6
    and (
      p.pos_timestamp >= TIMESTAMP '{{start_date}}' - interval '330' minute
      and p.pos_timestamp < TIMESTAMP '{{end_date}}' - interval '330' minute + interval '1' day
    )
    and p.insert_ds_ist >= '{{start_date}}'
    and p.insert_ds_ist <= cast(date '{{end_date}}' + interval '1' day as varchar)
    and upper(trim(cti.legal_name)) = 'BLINK COMMERCE PRIVATE LIMITED'
    
  GROUP BY 1,2,3,4,5,6
'''

print(RECALL_ASSISTANCE_PRN_SQL[:300], '...')

In [ ]:
RECALL_COLUMNS = ['duration', 'manufacturer_id', 'manufacturer', 'item_id', 'item_name', 'entity', 'prn_qty', 'recall_charges']

if should_run('recall_assistance'):
    _recall_result = run_date_batched(
        label='recall_assistance', sql_template=RECALL_ASSISTANCE_PRN_SQL, columns=RECALL_COLUMNS,
        parquet_path=PARQUET_FILES['recall_assistance'], batch_days=BATCH_DAYS_RECALL,
        numeric_cols=['prn_qty', 'recall_charges'],
        date_cols=['duration'],
        extra_params={'day_week_month': DAY_WEEK_MONTH},
    )
    if _recall_result['rows'] <= 100:
        log('WARNING: total rows <= 100 across the whole extract - check the LIMIT 100 removal took effect.', label='recall_assistance')
    RUN_RESULTS['recall_assistance'] = _recall_result
else:
    log('SKIPPED (RUN_FLAGS set to No)', label='recall_assistance')

## 10. Query 7 — IOCC Early Payment Discount (BCPL)

Batched by **date window** — period-scoped, hits invoice/payment tables. Output:
`PARQUET_FILES['early_payment']`. Source query had no upper date bound at all; added
`AND i.insert_ds_ist <= '{{end_date}}'` to make batching possible (flagged for
confirmation — see `queries_repository.md` entry 007). Also dropped a pointless
`ORDER BY payment_date DESC` with no `LIMIT`.

Controlled by `RUN_FLAGS['early_payment']` (set to `'No'` in CONFIG to skip).

In [ ]:
EARLY_PAYMENT_DISCOUNT_SQL = r'''WITH 
payments AS (
  SELECT
    i.id                                           AS invoice_id,
    i.vendor_invoice_id                            AS vendor_invoice_id,
    i.invoice_date,
    i.grn_date,
    ipm.payment_date,
    ipm.payment_amount,
    CAST(
      json_query(i.meta, 'lax $.credit_days' omit quotes)
      AS integer
    )                                              AS credit_days,
    po.vendor_id,
    v.vendor_name
  FROM po.invoice i
  JOIN po.invoice_payment_mapping ipm
    ON i.id = ipm.invoice_id
  JOIN po.purchase_order po
    ON po.po_number = i.order_number
  LEFT JOIN vms.vms_vendor v
    ON v.id = po.vendor_id
  WHERE 
    i.insert_ds_ist   >= '{{start_date}}'
    AND i.insert_ds_ist <= '{{end_date}}'
    AND i.invoice_status IN (4,5,6)
),
early_payment AS (
  SELECT
    invoice_id,
    vendor_invoice_id,
    invoice_date,
    grn_date,
    payment_date,                                
    credit_days,                                 
    vendor_name         AS vendor,
    payment_amount      AS total_amount,
    CASE
      WHEN payment_date <= DATE_ADD('day', 1, invoice_date)
      THEN payment_amount * 0.18 * (credit_days - 1) / 365
      ELSE 0
    END                 AS interest_receivable
  FROM payments
)
SELECT
  date(payment_date) as payment_date,                               
  date(invoice_date) as invoice_date,
  date(grn_date) as grn_date,
  credit_days,
  vendor,
  invoice_id,
  vendor_invoice_id,
  total_amount,
  interest_receivable
FROM early_payment
WHERE interest_receivable > 0
'''

print(EARLY_PAYMENT_DISCOUNT_SQL[:300], '...')

In [ ]:
EARLYPAY_COLUMNS = ['payment_date', 'invoice_date', 'grn_date', 'credit_days', 'vendor', 'invoice_id', 'vendor_invoice_id', 'total_amount', 'interest_receivable']

if should_run('early_payment'):
    RUN_RESULTS['early_payment'] = run_date_batched(
        label='early_payment', sql_template=EARLY_PAYMENT_DISCOUNT_SQL, columns=EARLYPAY_COLUMNS,
        parquet_path=PARQUET_FILES['early_payment'], batch_days=BATCH_DAYS_EARLYPAY,
        numeric_cols=['credit_days', 'total_amount', 'interest_receivable'],
        date_cols=['payment_date', 'invoice_date', 'grn_date'],
    )
else:
    log('SKIPPED (RUN_FLAGS set to No)', label='early_payment')


## 11. Query 8 — IOCC Fill Rate Penalty

Batched by **date window** — period-scoped, hits PO/GRN tables. Output:
`PARQUET_FILES['fill_rate_penalty']`. Uses `{{start}}`/`{{end}}` placeholder names (not
`{{start_date}}`/`{{end_date}}`). Note: despite the column name `day_week_month`, this
query's period grain is hardcoded to month-level — there's no `{{day_week_month}}`
placeholder here. Also dropped a no-op `ORDER BY 9 desc` on an intermediate CTE.

Controlled by `RUN_FLAGS['fill_rate_penalty']` (set to `'No'` in CONFIG to skip).

In [ ]:
FILL_RATE_PENALTY_SQL = r'''with base as(
select 
        day_week_month,
        po_number,
        sender_vendor_id,
        sender_vendor_name,
        destination_vendor,
        po.item_id, 
        po.item_name,
        po.outlet_id,
        po.outlet_name, 
        is_multiple_grn,
        po.manufacturer_name as mfr_name,
        cast(manufacturer_id as varchar)  as mfr_id,

        sum(po.po_units_ordered) po_qty_ordered, 
        sum(po.po_units_ordered*po.po_landing_price) po_amt_ordered,
        -- sum(stockin.stockin_quantity) stockin_quantity,
        sum(grn_qty) po_qty_received,
        sum(grn_qty*po.po_landing_price) po_amt_received

from
(
    select 
        po.outlet_id, 
        co.name as outlet_name, 
        case when business_type_id = 7 then 'DS' else 'WH' end as outlet_type,
        cl.name as city,
        po.vendor_id as sender_vendor_id, 
        po.vendor_name as sender_vendor_name,
        v.vendor_name as destination_vendor,
        po.po_number,
        is_multiple_grn,
        date(po.created_at) as po_date,
        daTE(PO.EXPIRY_DATE) AS EXPIRY_DATE,
        date(date_trunc('month', date(po.expiry_date))) as day_week_month,
        poi.item_id, 
        poi.name as item_name, 
        -- poi.upc, 
        poi.variant_id,
        poi.manufacturer_name, 
        -- poi.brand,
        poi.units_ordered as po_units_ordered, 
        poi.margin_percentage as po_margin,
        poi.landing_rate as po_landing_price, 
        poi.mrp as po_mrp, 
        poi.manufacturer_id,
        coalesce(grn.quantity,0) as grn_qty
    
    from po.purchase_order po
    inner join po.purchase_order_items poi on po.id = poi.po_id and poi.active = 1 and poi.lake_active_record = true
                                            and po.lake_active_record = true
    left join po.purchase_order_status os on os.po_id = po.id and os.lake_active_record = true
    join vms.vms_vendor vv on vv.id=po.vendor_id and vv.type_id=3
    inner join retail.console_outlet co on po.outlet_id = co.id and co.lake_active_record = true
    join retail.console_location cl on cl.id = co.tax_location_id and cl.lake_active_record = true
    LEFT JOIN vms.vms_vendor v ON v.id = po.destination_entity_vendor_id and v.lake_active_record = true
    left join (select item_id, coalesce(handling_type,'0') as handling_type from rpc.product_product group by 1,2) h on h.item_id = poi.item_id
    left join (select item_id, po_id, insert_ds_ist, sum(quantity) quantity from po.po_grn group by 1,2,3) grn on grn.item_id = poi.item_id and grn.po_id = po.id 
                        and grn.insert_ds_ist between cast(date'{{start}}' - interval '30' day as varchar) and cast(date'{{end}}' + interval '60' day as varchar)
    
    where 

        po.expiry_date between timestamp'{{start}}' - interval '330' minute and timestamp'{{end}}' - interval '330' minute
        and po.vendor_name not like 'Dummy %%'
        and poi.landing_rate <> 0
        and handling_type <> '8'
        and po_state_id  in (9,     --fulfilled
                                8     --Expired
                                )

) po

where day_week_month between date'{{start}}' and date'{{end}}'
-- and sender_vendor_id = 25759
group by 1,2,3,4,5,6,7,8,9,10,11,12
),


contract_details as(
WITH mapping_agg AS (
    SELECT
        cm.contract_id,
        ARRAY_AGG(
            CAST(
                ROW(cm.mapping_type, cm.id)
                AS ROW(mapping_type VARCHAR, ref_id BIGINT)
            )
        ) FILTER (WHERE cm.id IS NOT NULL) AS mappings,
        ARRAY_AGG(cm.mapped_id)
            FILTER (WHERE cm.mapping_type = 'MANUFACTURER') AS manufacturer_ids,
        ARRAY_AGG(
            cm.mapped_value
        ) FILTER (WHERE cm.mapping_type = 'MANUFACTURER') AS manufacturer_names

    FROM clm.clm_contract_mapping cm
    WHERE cm.is_deleted = 0
    GROUP BY cm.contract_id
),
raw as (
SELECT
    c.id AS contract_id,
    c.state AS contract_state,
    c.source_company_id AS company_purchaser_id,
    c.source_company_name  AS company_purchaser_name,
    c.destination_company_id AS company_buyer_id,
    c.destination_company_name AS company_buyer_name,
    s.name AS section_name,
    qa.id AS question_answer_id,
    qa.question_id,
    -- qa.answer,
    json_parse(qa.answer) as answer,
    ma.mappings,
    ma.manufacturer_ids,
    ma.manufacturer_names,
    c.execution_date,start_date as effective_date,
    case when c.end_date is null then 'Yes' else 'No' end as valid_until_cancelled,
c.updated_at as contract_updated_at
FROM clm.clm_contract c
LEFT JOIN clm.clm_contract_section cs
       ON cs.contract_id = c.id
      AND cs.is_deleted = 0
LEFT JOIN clm.clm_section s
       ON s.id = cs.section_id
      AND s.is_deleted = 0
LEFT JOIN clm.clm_section_question_answer qa
       ON qa.contract_section_id = cs.id
      AND qa.is_deleted = 0
LEFT JOIN mapping_agg ma
       ON ma.contract_id = c.id
WHERE c.is_deleted = 0
and c.state not in ('ARCHIVED','REJECTED')
ORDER BY
    cs.section_id NULLS LAST,
    qa.question_id NULLS LAST,
    qa.id NULLS LAST

),

format_ as (
select contract_id,
case when json_extract_scalar(x, '$.Manufacturer.id') is not null then 
json_extract_scalar(x, '$.Manufacturer.id') else cast(manufacturer_ids[1] as varchar) end
AS manufacturer_id,
case when json_extract_scalar(x, '$.Manufacturer.name') is not null then
json_extract_scalar(x, '$.Manufacturer.name') else cast(manufacturer_names[1] as varchar) end
AS manufacturer,
company_buyer_id,
company_buyer_name,
contract_state,
question_id,
json_extract_scalar(x,'$.value') as answer_values
from
raw
cross join unnest (CASE 
        WHEN json_extract(answer, '$.values') IS NOT NULL 
        THEN CAST(json_extract(answer, '$.values') AS array(json))
        ELSE ARRAY[answer]
    END 
) as t(x)
),


fill_rate as (
select contract_id,
manufacturer_id,
manufacturer,
contract_state,
min(answer_values) filter(where question_id='type_of_purchase_v1') as type_of_purchase,
min(answer_values) filter(where question_id='fill_rate_enable_v1') as fill_rate_penalty,
min(answer_values) filter(where question_id='fill_rate_threshold_percent_v1') as fill_rate_threshold_percent,
min(answer_values) filter(where question_id='fill_rate_penalty_rs_v1') as fill_rate_penalty_percent

from format_
group by 1,2,3,4
)

select * from 
fill_rate where fill_rate_penalty='Yes'
),


fff as (
select
    day_week_month,
    -- cast(po_number as varchar) po_number,
    cast(sender_vendor_id as varchar) as sender_vendor_id,
    sender_vendor_name,
    destination_vendor,
    mfr_id,
    mfr_name,
    is_multiple_grn,
    cast(fill_rate_threshold_percent as int)  as fill_rate_threshold_percent,
    cast(fill_rate_penalty_percent as int) as fill_rate_penalty_percent,
    sum(po_qty_ordered) po_qty_ordered,
    sum(po_qty_received) po_qty_received,
    -- sum(stockin_quantity) stockin_quantity,
    
    sum(po_qty_received)*100/sum(po_qty_ordered) as fill_rate,
    sum(po_amt_ordered) po_amt_ordered,
    sum(po_amt_received) po_amt_received
from base b
join contract_details cd on cd.manufacturer_id=b.mfr_id
where po_qty_ordered <> 0
group by 1,2,3,4,5,6,7,8,9
)

select *,
case when fill_rate<fill_rate_threshold_percent
then (fill_rate_penalty_percent * (po_amt_ordered-po_amt_received))/100
else 0 end as penalty
from fff
where fill_rate<fill_rate_threshold_percent
and fill_rate_penalty_percent>0
'''

print(FILL_RATE_PENALTY_SQL[:300], '...')

In [ ]:
FILLRATE_COLUMNS = [
    'day_week_month', 'sender_vendor_id', 'sender_vendor_name', 'destination_vendor',
    'mfr_id', 'mfr_name', 'is_multiple_grn', 'fill_rate_threshold_percent',
    'fill_rate_penalty_percent', 'po_qty_ordered', 'po_qty_received', 'fill_rate',
    'po_amt_ordered', 'po_amt_received', 'penalty',
]

if should_run('fill_rate_penalty'):
    RUN_RESULTS['fill_rate_penalty'] = run_date_batched(
        label='fill_rate_penalty', sql_template=FILL_RATE_PENALTY_SQL, columns=FILLRATE_COLUMNS,
        parquet_path=PARQUET_FILES['fill_rate_penalty'], batch_days=BATCH_DAYS_FILLRATE,
        numeric_cols=[
            'fill_rate_threshold_percent', 'fill_rate_penalty_percent', 'po_qty_ordered',
            'po_qty_received', 'fill_rate', 'po_amt_ordered', 'po_amt_received', 'penalty',
        ],
        date_cols=['day_week_month'],
        extra_params={'start': START_DATE, 'end': END_DATE},
    )
else:
    log('SKIPPED (RUN_FLAGS set to No)', label='fill_rate_penalty')


## 12. Query 9 — IOCC Storage Fees (aligned across brands)

Batched by **contract_id range** (current-snapshot). Output: `PARQUET_FILES['storage_fees']`.
Note: this query filters `state IN ('APPROVED','PENDING APPROVAL')` — a whitelist, unlike
most other contract-terms queries' blacklist — so contract coverage here is narrower than
e.g. `contracts_extract.parquet`.

Controlled by `RUN_FLAGS['storage_fees']` (set to `'No'` in CONFIG to skip).

In [ ]:
STORAGE_FEES_SQL = r'''WITH mapping_agg AS (
    SELECT
        cm.contract_id,
        ARRAY_AGG(
            CAST(
                ROW(cm.mapping_type, cm.id)
                AS ROW(mapping_type VARCHAR, ref_id BIGINT)
            )
        ) FILTER (WHERE cm.id IS NOT NULL) AS mappings,
        ARRAY_AGG(cm.mapped_id)
            FILTER (WHERE cm.mapping_type = 'MANUFACTURER') AS manufacturer_ids,
        ARRAY_AGG(
            cm.mapped_value
        ) FILTER (WHERE cm.mapping_type = 'MANUFACTURER') AS manufacturer_names
    FROM clm.clm_contract_mapping cm
    WHERE cm.is_deleted = 0
    GROUP BY cm.contract_id
),
raw as (
SELECT
    c.id                               AS contract_id,
    c.state                            AS contract_state,
    c.created_at,
    c.created_by,
    c.updated_at,
    c.updated_by,
    c.source_company_id              AS company_purchaser_id,
    c.source_company_name            AS company_purchaser_name,
    c.destination_company_id                  AS company_buyer_id,
    c.destination_company_name                AS company_buyer_name,
    s.name                             AS section_name,
    qa.id                              AS question_answer_id,
    qa.question_id,
    -- qa.answer,
    json_parse(qa.answer) as answer,
    qa.remarks,
    qa.created_at                      AS qa_created_at,
    qa.created_by                      AS qa_created_by,
    ma.mappings,
    ma.manufacturer_ids,
    ma.manufacturer_names
FROM clm.clm_contract c
LEFT JOIN clm.clm_contract_section cs
       ON cs.contract_id = c.id
      AND cs.is_deleted = 0
JOIN clm.clm_section s
       ON s.id = cs.section_id
      AND s.is_deleted = 0 and s.name = 'Storage Fees'
LEFT JOIN clm.clm_section_question_answer qa
       ON qa.contract_section_id = cs.id
      AND qa.is_deleted = 0
LEFT JOIN mapping_agg ma
       ON ma.contract_id = c.id
WHERE c.is_deleted = 0
and c.state  in ('APPROVED','PENDING APPROVAL')
and c.id BETWEEN {{id_from}} AND {{id_to}}
ORDER BY
    cs.section_id NULLS LAST,
    qa.question_id NULLS LAST,
    qa.id NULLS LAST
),
format_ as (
select contract_id,
case when json_extract_scalar(x, '$.Manufacturer.id') is not null then 
json_extract_scalar(x, '$.Manufacturer.id') else cast(manufacturer_ids[1] as varchar) end
AS manufacturer_id,
case when json_extract_scalar(x, '$.Manufacturer.name') is not null then
json_extract_scalar(x, '$.Manufacturer.name') else cast(manufacturer_names[1] as varchar) end
AS manufacturer,
contract_state,
created_at,
created_by,
updated_at,
updated_by,
company_purchaser_id,
company_purchaser_name,
company_buyer_id,
company_buyer_name,
section_name,
question_answer_id,
question_id,
answer,
json_extract_scalar(x,'$.value') as answer_values,
x
from 
raw
cross join unnest (CASE 
        WHEN json_extract(answer, '$.values') IS NOT NULL 
        THEN CAST(json_extract(answer, '$.values') AS array(json))
        ELSE ARRAY[answer]
    END 
) as t(x)
),
details as (select contract_id,
json_extract_scalar(row, '$.rule_op') AS rule,
json_extract_scalar(row, '$.start_day') AS start_day,
CAST(json_extract_scalar(row, '$.sku_gte_1_cuft') AS double) AS sku_gte_1_cuft,
CAST(json_extract_scalar(row, '$.sku_lt_1_cuft') AS double) AS sku_lt_1_cuft from format_ 
cross join unnest(
     CAST(json_extract(x, '$.value.rows') AS array(json)) ) AS t(row)
where question_id='sale_window_details_v1'
),
final as (select f.contract_id,manufacturer_id,
manufacturer,contract_state,
min(answer_values) filter(where question_id='unsold_storage_fees_enable_v1') as storage_fee,
rule,
start_day,
sku_gte_1_cuft,
sku_lt_1_cuft
from format_ f
left join details d on f.contract_id=d.contract_id
group by 1,2,3,4,6,7,8,9
)
select * from final where storage_fee='Yes'
'''

print(STORAGE_FEES_SQL[:300], '...')

In [ ]:
STORAGE_COLUMNS = [
    'contract_id', 'manufacturer_id', 'manufacturer', 'contract_state', 'storage_fee',
    'rule', 'start_day', 'sku_gte_1_cuft', 'sku_lt_1_cuft',
]
STORAGE_BOUNDS_SQL = '''
SELECT MIN(id) AS min_id, MAX(id) AS max_id, COUNT(*) AS n_contracts
FROM clm.clm_contract
WHERE is_deleted = 0
  AND state IN ('APPROVED','PENDING APPROVAL')
'''

if should_run('storage_fees'):
    RUN_RESULTS['storage_fees'] = run_id_batched(
        label='storage_fees', sql_template=STORAGE_FEES_SQL, columns=STORAGE_COLUMNS,
        parquet_path=PARQUET_FILES['storage_fees'], bounds_sql=STORAGE_BOUNDS_SQL,
        batch_size=BATCH_SIZE_STORAGE,
        numeric_cols=['sku_gte_1_cuft', 'sku_lt_1_cuft'],
    )
else:
    log('SKIPPED (RUN_FLAGS set to No)', label='storage_fees')


## 13. Query 10 — IOCC Aligned RTV Days for OR and SOR Brands

Batched by **contract_id range** (current-snapshot). Output: `PARQUET_FILES['rtv_days']`.
Same `APPROVED`/`PENDING APPROVAL` whitelist as query 9. Note: this query's JSON unnest
checks `$.Values` (capital V) unlike the lowercase `$.values` used everywhere else — kept
as given since the true JSON key casing for this section isn't verifiable from here.

Controlled by `RUN_FLAGS['rtv_days']` (set to `'No'` in CONFIG to skip).

In [ ]:
RTV_DAYS_SQL = r'''WITH mapping_agg AS (
    SELECT
        cm.contract_id,
        ARRAY_AGG(
            CAST(
                ROW(cm.mapping_type, cm.id)
                AS ROW(mapping_type VARCHAR, ref_id BIGINT)
            )
        ) FILTER (WHERE cm.id IS NOT NULL) AS mappings,
        ARRAY_AGG(cm.mapped_id)
            FILTER (WHERE cm.mapping_type = 'MANUFACTURER') AS manufacturer_ids,
        ARRAY_AGG(
            cm.mapped_value
        ) FILTER (WHERE cm.mapping_type = 'MANUFACTURER') AS manufacturer_names
    FROM clm.clm_contract_mapping cm
    WHERE cm.is_deleted = 0
    GROUP BY cm.contract_id
),
raw as (
SELECT
    c.id                               AS contract_id,
    c.state                            AS contract_state,
    c.created_at,
    c.created_by,
    c.updated_at,
    c.updated_by,
    c.source_company_id              AS company_purchaser_id,
    c.source_company_name            AS company_purchaser_name,
    c.destination_company_id                  AS company_buyer_id,
    c.destination_company_name                AS company_buyer_name,
    s.name                             AS section_name,
    qa.id                              AS question_answer_id,
    qa.question_id,
    json_parse(qa.answer) as answer,
    qa.remarks,
    qa.created_at                      AS qa_created_at,
    qa.created_by                      AS qa_created_by,
    ma.mappings,
    ma.manufacturer_ids,
    ma.manufacturer_names,
    c.execution_date,start_date as effective_date,
    case when c.end_date is null then 'Yes' else 'No' end as valid_until_cancelled,
c.updated_at as contract_updated_at
FROM clm.clm_contract c
LEFT JOIN clm.clm_contract_section cs
       ON cs.contract_id = c.id
      AND cs.is_deleted = 0
LEFT JOIN clm.clm_section s
       ON s.id = cs.section_id
      AND s.is_deleted = 0
LEFT JOIN clm.clm_section_question_answer qa
       ON qa.contract_section_id = cs.id
      AND qa.is_deleted = 0
LEFT JOIN mapping_agg ma
       ON ma.contract_id = c.id
WHERE c.is_deleted = 0
and s.name='Sale-Purchase Model'
and c.state in ('PENDING APPROVAL','APPROVED')
and c.id BETWEEN {{id_from}} AND {{id_to}}
-- and qa.question_id in ('type_of_purchase_v1','rtv_days_v1')
ORDER BY
    cs.section_id NULLS LAST,
    qa.question_id NULLS LAST,
    qa.id NULLS LAST
),
format_ as (
select contract_id,
case when json_extract_scalar(x, '$.Manufacturer.id') is not null then 
json_extract_scalar(x, '$.Manufacturer.id') else cast(manufacturer_ids[1] as varchar) end
AS manufacturer_id,
case when json_extract_scalar(x, '$.Manufacturer.name') is not null then
json_extract_scalar(x, '$.Manufacturer.name') else cast(manufacturer_names[1] as varchar) end
AS manufacturer,
section_name,
question_id,
contract_state,
case when json_extract_scalar(x,'$.l2.name') is null then 'All' else json_extract_scalar(x,'$.l2.name') end as level,
json_extract_scalar(x,'$.value') as answer_values
from
raw
cross join
UNNEST (
    CASE
        WHEN answer IS NOT NULL AND json_extract(answer, '$.Values') IS NOT NULL
        THEN CAST(json_extract(answer, '$.Values') AS ARRAY(JSON))
        WHEN answer IS NOT NULL
        THEN ARRAY[answer]
        ELSE ARRAY[]
    END) as t(x)
    )
    
    
select contract_id,
manufacturer_id,
manufacturer,
contract_state,
level,
max(case when question_id='type_of_purchase_v1' then answer_values else null end ) as type_of_purchase,
max(case when question_id='rtv_days_v1' then answer_values else null end ) as rtv_days,
max(case when question_id='sor_days_v1' then answer_values else null end ) as sor_days
from format_
group by 1,2,3,4,5
'''

print(RTV_DAYS_SQL[:300], '...')

In [ ]:
RTV_COLUMNS = ['contract_id', 'manufacturer_id', 'manufacturer', 'contract_state', 'level', 'type_of_purchase', 'rtv_days', 'sor_days']
RTV_BOUNDS_SQL = '''
SELECT MIN(id) AS min_id, MAX(id) AS max_id, COUNT(*) AS n_contracts
FROM clm.clm_contract
WHERE is_deleted = 0
  AND state IN ('PENDING APPROVAL','APPROVED')
'''

if should_run('rtv_days'):
    RUN_RESULTS['rtv_days'] = run_id_batched(
        label='rtv_days', sql_template=RTV_DAYS_SQL, columns=RTV_COLUMNS,
        parquet_path=PARQUET_FILES['rtv_days'], bounds_sql=RTV_BOUNDS_SQL,
        batch_size=BATCH_SIZE_RTV,
    )
else:
    log('SKIPPED (RUN_FLAGS set to No)', label='rtv_days')


## 14. Query 11 — IOCC Minimum Ads Spend by Manufacturer

Batched by **contract_id range** (current-snapshot). Output: `PARQUET_FILES['ads_spend']`.

Controlled by `RUN_FLAGS['ads_spend']` (set to `'No'` in CONFIG to skip).

In [ ]:
ADS_SPEND_SQL = r'''WITH mapping_agg AS (
    SELECT
        cm.contract_id,
        ARRAY_AGG(
            CAST(
                ROW(cm.mapping_type, cm.id)
                AS ROW(mapping_type VARCHAR, ref_id BIGINT)
            )
        ) FILTER (WHERE cm.id IS NOT NULL) AS mappings,
        ARRAY_AGG(cm.mapped_id)
            FILTER (WHERE cm.mapping_type = 'MANUFACTURER') AS manufacturer_ids,
        ARRAY_AGG(
            cm.mapped_value
        ) FILTER (WHERE cm.mapping_type = 'MANUFACTURER') AS manufacturer_names
    FROM clm.clm_contract_mapping cm
    WHERE cm.is_deleted = 0
    GROUP BY cm.contract_id
),
raw as (
SELECT
    c.id                               AS contract_id,
    c.state                            AS contract_state,
    c.created_at,
    c.created_by,
    c.updated_at,
    c.updated_by,
    c.source_company_id              AS company_purchaser_id,
    c.source_company_name            AS company_purchaser_name,
    c.destination_company_id                  AS company_buyer_id,
    c.destination_company_name                AS company_buyer_name,
    s.name                             AS section_name,
    qa.id                              AS question_answer_id,
    qa.question_id,
    -- qa.answer,
    json_parse(qa.answer) as answer,
    qa.remarks,
    qa.created_at                      AS qa_created_at,
    qa.created_by                      AS qa_created_by,
    ma.mappings,
    ma.manufacturer_ids,
    ma.manufacturer_names,
    c.execution_date,start_date as effective_date,
    case when c.end_date is null then 'Yes' else 'No' end as valid_until_cancelled,
c.updated_at as contract_updated_at
FROM clm.clm_contract c
LEFT JOIN clm.clm_contract_section cs
       ON cs.contract_id = c.id
      AND cs.is_deleted = 0
LEFT JOIN clm.clm_section s
       ON s.id = cs.section_id
      AND s.is_deleted = 0
LEFT JOIN clm.clm_section_question_answer qa
       ON qa.contract_section_id = cs.id
      AND qa.is_deleted = 0
LEFT JOIN mapping_agg ma
       ON ma.contract_id = c.id
WHERE c.is_deleted = 0
and c.state not in ('ARCHIVED','REJECTED')
and c.id BETWEEN {{id_from}} AND {{id_to}}
ORDER BY
    cs.section_id NULLS LAST,
    qa.question_id NULLS LAST,
    qa.id NULLS LAST
),
format_ as (
select contract_id,
case when json_extract_scalar(x, '$.Manufacturer.id') is not null then 
json_extract_scalar(x, '$.Manufacturer.id') else cast(manufacturer_ids[1] as varchar) end
AS manufacturer_id,
case when json_extract_scalar(x, '$.Manufacturer.name') is not null then
json_extract_scalar(x, '$.Manufacturer.name') else cast(manufacturer_names[1] as varchar) end
AS manufacturer,
-- company_purchaser_id,
-- company_purchaser_name,
company_buyer_id,
company_buyer_name,
contract_state,
created_at,
created_by,
updated_at,
updated_by,
execution_date,
effective_date,
valid_until_cancelled,
section_name,
question_id,
question_answer_id,
contract_updated_at,
json_extract_scalar(x,'$.value') as answer_values
from
raw
cross join unnest (CASE 
        WHEN json_extract(answer, '$.values') IS NOT NULL 
        THEN CAST(json_extract(answer, '$.values') AS array(json))
        ELSE ARRAY[answer]
    END 
) as t(x)
)
select 
contract_id,
manufacturer_id,
manufacturer,
contract_state,
min(answer_values) filter(where question_id='credit_days_ads_business_v1') as credit_days_ads_business,
min(answer_values) filter(where question_id='min_ads_spend_enable_v1') as min_ads_spend_enabled,
min(answer_values) filter(where question_id='min_ads_spend_value_percent_v1') as min_ads_spend_value_percent,
min(answer_values) filter(where question_id='min_ads_basis_v1') as min_ads_basis
from format_
group by 1,2,3,4
'''

print(ADS_SPEND_SQL[:300], '...')

In [ ]:
ADSPEND_COLUMNS = ['contract_id', 'manufacturer_id', 'manufacturer', 'contract_state', 'credit_days_ads_business', 'min_ads_spend_enabled', 'min_ads_spend_value_percent', 'min_ads_basis']
ADSPEND_BOUNDS_SQL = '''
SELECT MIN(id) AS min_id, MAX(id) AS max_id, COUNT(*) AS n_contracts
FROM clm.clm_contract
WHERE is_deleted = 0
  AND state NOT IN ('ARCHIVED','REJECTED')
'''

if should_run('ads_spend'):
    RUN_RESULTS['ads_spend'] = run_id_batched(
        label='ads_spend', sql_template=ADS_SPEND_SQL, columns=ADSPEND_COLUMNS,
        parquet_path=PARQUET_FILES['ads_spend'], bounds_sql=ADSPEND_BOUNDS_SQL,
        batch_size=BATCH_SIZE_ADSPEND,
    )
else:
    log('SKIPPED (RUN_FLAGS set to No)', label='ads_spend')


## 15. Query 12 — IOCC Turnover Discount by Manufacturer

Batched by **contract_id range** (current-snapshot). Output: `PARQUET_FILES['turnover_discount']`.
Source query's `"Target based incentives"` alias renamed to `target_based_incentives`
(matches the same field's name in `contracts_extract.parquet`).

Controlled by `RUN_FLAGS['turnover_discount']` (set to `'No'` in CONFIG to skip).

In [ ]:
TURNOVER_DISCOUNT_SQL = r'''WITH mapping_agg AS (
    SELECT
        cm.contract_id,
        ARRAY_AGG(
            CAST(
                ROW(cm.mapping_type, cm.id)
                AS ROW(mapping_type VARCHAR, ref_id BIGINT)
            )
        ) FILTER (WHERE cm.id IS NOT NULL) AS mappings,
        ARRAY_AGG(cm.mapped_id)
            FILTER (WHERE cm.mapping_type = 'MANUFACTURER') AS manufacturer_ids,
        ARRAY_AGG(
            cm.mapped_value
        ) FILTER (WHERE cm.mapping_type = 'MANUFACTURER') AS manufacturer_names
    FROM clm.clm_contract_mapping cm
    WHERE cm.is_deleted = 0
    GROUP BY cm.contract_id
),
raw as (
SELECT
    c.id                               AS contract_id,
    c.state                            AS contract_state,
    c.created_at,
    c.created_by,
    c.updated_at,
    c.updated_by,
    c.source_company_id              AS company_purchaser_id,
    c.source_company_name            AS company_purchaser_name,
    c.destination_company_id                  AS company_buyer_id,
    c.destination_company_name                AS company_buyer_name,
    s.name                             AS section_name,
    qa.id                              AS question_answer_id,
    qa.question_id,
    -- qa.answer,
    json_parse(qa.answer) as answer,
    qa.remarks,
    qa.created_at                      AS qa_created_at,
    qa.created_by                      AS qa_created_by,
    ma.mappings,
    ma.manufacturer_ids,
    ma.manufacturer_names,
    c.execution_date,start_date as effective_date,
    case when c.end_date is null then 'Yes' else 'No' end as valid_until_cancelled,
c.updated_at as contract_updated_at
FROM clm.clm_contract c
LEFT JOIN clm.clm_contract_section cs
       ON cs.contract_id = c.id
      AND cs.is_deleted = 0
LEFT JOIN clm.clm_section s
       ON s.id = cs.section_id
      AND s.is_deleted = 0
LEFT JOIN clm.clm_section_question_answer qa
       ON qa.contract_section_id = cs.id
      AND qa.is_deleted = 0
LEFT JOIN mapping_agg ma
       ON ma.contract_id = c.id
WHERE c.is_deleted = 0
and c.state not in ('ARCHIVED','REJECTED')
and c.id BETWEEN {{id_from}} AND {{id_to}}
ORDER BY
    cs.section_id NULLS LAST,
    qa.question_id NULLS LAST,
    qa.id NULLS LAST
),
format_ as (
select contract_id,
case when json_extract_scalar(x, '$.Manufacturer.id') is not null then 
json_extract_scalar(x, '$.Manufacturer.id') else cast(manufacturer_ids[1] as varchar) end
AS manufacturer_id,
case when json_extract_scalar(x, '$.Manufacturer.name') is not null then
json_extract_scalar(x, '$.Manufacturer.name') else cast(manufacturer_names[1] as varchar) end
AS manufacturer,
-- company_purchaser_id,
-- company_purchaser_name,
company_buyer_id,
company_buyer_name,
contract_state,
created_at,
created_by,
updated_at,
updated_by,
execution_date,
effective_date,
valid_until_cancelled,
section_name,
question_id,
question_answer_id,
contract_updated_at,
json_extract_scalar(x,'$.value') as answer_values
from
raw
cross join unnest (CASE 
        WHEN json_extract(answer, '$.values') IS NOT NULL 
        THEN CAST(json_extract(answer, '$.values') AS array(json))
        ELSE ARRAY[answer]
    END 
) as t(x)
)
select contract_id,
manufacturer_id,
manufacturer,
contract_state,
min(answer_values) filter(where question_id='tbi_enable_v1') as target_based_incentives,
min(answer_values) filter(where question_id='tbi_payout_freq_v1') as payout_frequency
from format_
group by 1,2,3,4
'''

print(TURNOVER_DISCOUNT_SQL[:300], '...')

In [ ]:
TURNOVER_COLUMNS = ['contract_id', 'manufacturer_id', 'manufacturer', 'contract_state', 'target_based_incentives', 'payout_frequency']
TURNOVER_BOUNDS_SQL = '''
SELECT MIN(id) AS min_id, MAX(id) AS max_id, COUNT(*) AS n_contracts
FROM clm.clm_contract
WHERE is_deleted = 0
  AND state NOT IN ('ARCHIVED','REJECTED')
'''

if should_run('turnover_discount'):
    RUN_RESULTS['turnover_discount'] = run_id_batched(
        label='turnover_discount', sql_template=TURNOVER_DISCOUNT_SQL, columns=TURNOVER_COLUMNS,
        parquet_path=PARQUET_FILES['turnover_discount'], bounds_sql=TURNOVER_BOUNDS_SQL,
        batch_size=BATCH_SIZE_TURNOVER,
    )
else:
    log('SKIPPED (RUN_FLAGS set to No)', label='turnover_discount')


## 16. Query 13 — IOCC ePOD Clause by Manufacturer

Batched by **contract_id range** (current-snapshot). Output: `PARQUET_FILES['epod_clause']`.

Controlled by `RUN_FLAGS['epod_clause']` (set to `'No'` in CONFIG to skip).

In [ ]:
EPOD_CLAUSE_SQL = r'''WITH mapping_agg AS (
    SELECT
        cm.contract_id,
        ARRAY_AGG(
            CAST(
                ROW(cm.mapping_type, cm.id)
                AS ROW(mapping_type VARCHAR, ref_id BIGINT)
            )
        ) FILTER (WHERE cm.id IS NOT NULL) AS mappings,
        ARRAY_AGG(cm.mapped_id)
            FILTER (WHERE cm.mapping_type = 'MANUFACTURER') AS manufacturer_ids,
        ARRAY_AGG(
            cm.mapped_value
        ) FILTER (WHERE cm.mapping_type = 'MANUFACTURER') AS manufacturer_names
    FROM clm.clm_contract_mapping cm
    WHERE cm.is_deleted = 0
    GROUP BY cm.contract_id
),
raw as (
SELECT
    c.id                               AS contract_id,
    c.state                            AS contract_state,
    c.created_at,
    c.created_by,
    c.updated_at,
    c.updated_by,
    c.source_company_id              AS company_purchaser_id,
    c.source_company_name            AS company_purchaser_name,
    c.destination_company_id                  AS company_buyer_id,
    c.destination_company_name                AS company_buyer_name,
    s.name                             AS section_name,
    qa.id                              AS question_answer_id,
    qa.question_id,
    -- qa.answer,
    json_parse(qa.answer) as answer,
    qa.remarks,
    qa.created_at                      AS qa_created_at,
    qa.created_by                      AS qa_created_by,
    ma.mappings,
    ma.manufacturer_ids,
    ma.manufacturer_names,
    c.execution_date,start_date as effective_date,
    case when c.end_date is null then 'Yes' else 'No' end as valid_until_cancelled,
c.updated_at as contract_updated_at
FROM clm.clm_contract c
LEFT JOIN clm.clm_contract_section cs
       ON cs.contract_id = c.id
      AND cs.is_deleted = 0
LEFT JOIN clm.clm_section s
       ON s.id = cs.section_id
      AND s.is_deleted = 0
LEFT JOIN clm.clm_section_question_answer qa
       ON qa.contract_section_id = cs.id
      AND qa.is_deleted = 0
LEFT JOIN mapping_agg ma
       ON ma.contract_id = c.id
WHERE c.is_deleted = 0
and c.state not in ('ARCHIVED','REJECTED')
and c.id BETWEEN {{id_from}} AND {{id_to}}
ORDER BY
    cs.section_id NULLS LAST,
    qa.question_id NULLS LAST,
    qa.id NULLS LAST
),
format_ as (
select contract_id,
case when json_extract_scalar(x, '$.Manufacturer.id') is not null then 
json_extract_scalar(x, '$.Manufacturer.id') else cast(manufacturer_ids[1] as varchar) end
AS manufacturer_id,
case when json_extract_scalar(x, '$.Manufacturer.name') is not null then
json_extract_scalar(x, '$.Manufacturer.name') else cast(manufacturer_names[1] as varchar) end
AS manufacturer,
-- company_purchaser_id,
-- company_purchaser_name,
company_buyer_id,
company_buyer_name,
contract_state,
created_at,
created_by,
updated_at,
updated_by,
execution_date,
effective_date,
valid_until_cancelled,
section_name,
question_id,
question_answer_id,
contract_updated_at,
json_extract_scalar(x,'$.value') as answer_values
from
raw
cross join unnest (CASE 
        WHEN json_extract(answer, '$.values') IS NOT NULL 
        THEN CAST(json_extract(answer, '$.values') AS array(json))
        ELSE ARRAY[answer]
    END 
) as t(x)
)
select contract_id,
manufacturer_id,
manufacturer,
contract_state,
min(answer_values) filter(where question_id='discrepancy_timeline_outer_case_v1') as timeline_outer_case,
min(answer_values) filter(where question_id='discrepancy_timeline_inner_case_v1') as timeline_inner_case,
max(answer_values) filter(where question_id='seller_dispute_time_v1') as timeline_raise_dispute
from format_
group by 1,2,3,4
'''

print(EPOD_CLAUSE_SQL[:300], '...')

In [ ]:
EPOD_COLUMNS = ['contract_id', 'manufacturer_id', 'manufacturer', 'contract_state', 'timeline_outer_case', 'timeline_inner_case', 'timeline_raise_dispute']
EPOD_BOUNDS_SQL = '''
SELECT MIN(id) AS min_id, MAX(id) AS max_id, COUNT(*) AS n_contracts
FROM clm.clm_contract
WHERE is_deleted = 0
  AND state NOT IN ('ARCHIVED','REJECTED')
'''

if should_run('epod_clause'):
    RUN_RESULTS['epod_clause'] = run_id_batched(
        label='epod_clause', sql_template=EPOD_CLAUSE_SQL, columns=EPOD_COLUMNS,
        parquet_path=PARQUET_FILES['epod_clause'], bounds_sql=EPOD_BOUNDS_SQL,
        batch_size=BATCH_SIZE_EPOD,
    )
else:
    log('SKIPPED (RUN_FLAGS set to No)', label='epod_clause')


## 17. Query 14 — IOCC SP Benchmarking Clause by Manufacturer

Batched by **contract_id range** (current-snapshot). Output: `PARQUET_FILES['sp_benchmarking']`.

Controlled by `RUN_FLAGS['sp_benchmarking']` (set to `'No'` in CONFIG to skip).

In [ ]:
SP_BENCHMARKING_SQL = r'''WITH mapping_agg AS (
    SELECT
        cm.contract_id,
        ARRAY_AGG(
            CAST(
                ROW(cm.mapping_type, cm.id)
                AS ROW(mapping_type VARCHAR, ref_id BIGINT)
            )
        ) FILTER (WHERE cm.id IS NOT NULL) AS mappings,
        ARRAY_AGG(cm.mapped_id)
            FILTER (WHERE cm.mapping_type = 'MANUFACTURER') AS manufacturer_ids,
        ARRAY_AGG(
            cm.mapped_value
        ) FILTER (WHERE cm.mapping_type = 'MANUFACTURER') AS manufacturer_names
    FROM clm.clm_contract_mapping cm
    WHERE cm.is_deleted = 0
    GROUP BY cm.contract_id
),
raw as (
SELECT
    c.id                               AS contract_id,
    c.state                            AS contract_state,
    c.created_at,
    c.created_by,
    c.updated_at,
    c.updated_by,
    c.source_company_id              AS company_purchaser_id,
    c.source_company_name            AS company_purchaser_name,
    c.destination_company_id                  AS company_buyer_id,
    c.destination_company_name                AS company_buyer_name,
    s.name                             AS section_name,
    qa.id                              AS question_answer_id,
    qa.question_id,
    -- qa.answer,
    json_parse(qa.answer) as answer,
    qa.remarks,
    qa.created_at                      AS qa_created_at,
    qa.created_by                      AS qa_created_by,
    ma.mappings,
    ma.manufacturer_ids,
    ma.manufacturer_names,
    c.execution_date,start_date as effective_date,
    case when c.end_date is null then 'Yes' else 'No' end as valid_until_cancelled,
c.updated_at as contract_updated_at
FROM clm.clm_contract c
LEFT JOIN clm.clm_contract_section cs
       ON cs.contract_id = c.id
      AND cs.is_deleted = 0
LEFT JOIN clm.clm_section s
       ON s.id = cs.section_id
      AND s.is_deleted = 0
LEFT JOIN clm.clm_section_question_answer qa
       ON qa.contract_section_id = cs.id
      AND qa.is_deleted = 0
LEFT JOIN mapping_agg ma
       ON ma.contract_id = c.id
WHERE c.is_deleted = 0
and c.state not in ('ARCHIVED','REJECTED')
and c.id BETWEEN {{id_from}} AND {{id_to}}
ORDER BY
    cs.section_id NULLS LAST,
    qa.question_id NULLS LAST,
    qa.id NULLS LAST
),
format_ as (
select contract_id,
case when json_extract_scalar(x, '$.Manufacturer.id') is not null then 
json_extract_scalar(x, '$.Manufacturer.id') else cast(manufacturer_ids[1] as varchar) end
AS manufacturer_id,
case when json_extract_scalar(x, '$.Manufacturer.name') is not null then
json_extract_scalar(x, '$.Manufacturer.name') else cast(manufacturer_names[1] as varchar) end
AS manufacturer,
-- company_purchaser_id,
-- company_purchaser_name,
company_buyer_id,
company_buyer_name,
contract_state,
created_at,
created_by,
updated_at,
updated_by,
execution_date,
effective_date,
valid_until_cancelled,
section_name,
question_id,
question_answer_id,
contract_updated_at,
json_extract_scalar(x,'$.value') as answer_values
from
raw
cross join unnest (CASE 
        WHEN json_extract(answer, '$.values') IS NOT NULL 
        THEN CAST(json_extract(answer, '$.values') AS array(json))
        ELSE ARRAY[answer]
    END 
) as t(x)
)
select
contract_id,
manufacturer_id,
manufacturer,
contract_state,
min(answer_values) filter(where question_id='sp_benchmark_enable_v1') as sp_benchmarking,
min(answer_values) filter(where question_id='sp_benchmark_aligned_rm_percent_v1') as sp_benchmark_aligned_rm_percent
from format_
group by 1,2,3,4
'''

print(SP_BENCHMARKING_SQL[:300], '...')

In [ ]:
SPBENCH_COLUMNS = ['contract_id', 'manufacturer_id', 'manufacturer', 'contract_state', 'sp_benchmarking', 'sp_benchmark_aligned_rm_percent']
SPBENCH_BOUNDS_SQL = '''
SELECT MIN(id) AS min_id, MAX(id) AS max_id, COUNT(*) AS n_contracts
FROM clm.clm_contract
WHERE is_deleted = 0
  AND state NOT IN ('ARCHIVED','REJECTED')
'''

if should_run('sp_benchmarking'):
    RUN_RESULTS['sp_benchmarking'] = run_id_batched(
        label='sp_benchmarking', sql_template=SP_BENCHMARKING_SQL, columns=SPBENCH_COLUMNS,
        parquet_path=PARQUET_FILES['sp_benchmarking'], bounds_sql=SPBENCH_BOUNDS_SQL,
        batch_size=BATCH_SIZE_SPBENCH,
    )
else:
    log('SKIPPED (RUN_FLAGS set to No)', label='sp_benchmarking')


## 18. Query 15 — Manufacturer-Entity Monthly PRN Qty

Batched by **calendar month** — period-scoped via `month_range(START_DATE, END_DATE)`,
since this query takes a single `{{target_month}}` rather than a start/end range. Output:
`PARQUET_FILES['prn_monthly']`. Restricted to a fixed whitelist of 8 BCPL-group entity names.
Note: reads from `lake_`-prefixed tables, unlike query 6's unprefixed equivalents for
what looks like the same underlying data — kept as given, flagged in
`queries_repository.md` entry 015.

Controlled by `RUN_FLAGS['prn_monthly']` (set to `'No'` in CONFIG to skip).

In [ ]:
PRN_MONTHLY_SQL = r'''WITH month_range AS (
    SELECT 
        CAST('{{target_month}}-01' AS DATE) as start_date,
        date_add('month', 1, CAST('{{target_month}}-01' AS DATE)) as end_date
),
variants AS (
    SELECT DISTINCT 
           variant_id, 
           manufacturer_id
    FROM po.purchase_order_items 
),
unique_tax_info AS (
    SELECT 
        vendor_city_id, 
        legal_name,
        regexp_replace(upper(trim(legal_name)), '[^a-zA-Z0-9]', '') as normalized_name,
        ROW_NUMBER() OVER(PARTITION BY vendor_city_id ORDER BY id DESC) as rn
    FROM lake_vms.vms_vendor_city_tax_info
    WHERE active = 1
),
summary_base AS (
    SELECT 
        var.manufacturer_id AS "Manufacturer ID",
        p2.manufacturer AS "Manufacturer Name",
        upper(trim(cti.legal_name)) AS "Entity",
        CAST(SUM(p1.quantity) AS BIGINT) AS "Total PRN Quantity"
    FROM lake_pos.pos_invoice p
    CROSS JOIN month_range mr
    INNER JOIN lake_pos.pos_invoice_product_details p1 
        ON p.id = p1.invoice_id 
        AND p1.insert_ds_ist >= CAST(mr.start_date AS VARCHAR)
        AND p1.insert_ds_ist < CAST(mr.end_date AS VARCHAR)
    INNER JOIN lake_rpc.product_product p2 
        ON p2.variant_id = p1.variant_id
    LEFT JOIN variants var 
        ON p2.variant_id = var.variant_id
    LEFT JOIN lake_vms.vms_vendor_city_mapping cm 
        ON cm.vendor_id = p.source_entity_vendor_id AND cm.active = 1
    LEFT JOIN unique_tax_info cti 
        ON cti.vendor_city_id = cm.id AND cti.rn = 1
    WHERE p.invoice_type_id = 6 
      AND p.insert_ds_ist >= CAST(mr.start_date AS VARCHAR)
      AND p.insert_ds_ist < CAST(mr.end_date AS VARCHAR)
    GROUP BY 1, 2, 3
)
SELECT * FROM summary_base
WHERE "Manufacturer ID" IS NOT NULL
  AND (
       regexp_replace("Entity", '[^a-zA-Z0-9]', '') LIKE '%HANDSONTRADES%' OR 
       regexp_replace("Entity", '[^a-zA-Z0-9]', '') LIKE '%BLINKCOMMERCE%' OR 
       regexp_replace("Entity", '[^a-zA-Z0-9]', '') LIKE '%MOONSTONEVENTURES%' OR 
       regexp_replace("Entity", '[^a-zA-Z0-9]', '') LIKE '%SUPERWELLCOMTRADE%' OR 
       regexp_replace("Entity", '[^a-zA-Z0-9]', '') LIKE '%ZOMATOHYPERPURE%' OR 
       regexp_replace("Entity", '[^a-zA-Z0-9]', '') LIKE '%ASVAHRETAIL%' OR
       regexp_replace("Entity", '[^a-zA-Z0-9]', '') LIKE '%BCPL%' OR
       regexp_replace("Entity", '[^a-zA-Z0-9]', '') LIKE '%ZHPL%'
  )
ORDER BY "Manufacturer Name" ASC, "Entity" ASC
'''

print(PRN_MONTHLY_SQL[:300], '...')

In [ ]:
PRNMONTHLY_COLUMNS = ['target_month', 'manufacturer_id', 'manufacturer', 'entity', 'total_prn_qty']
prn_target_months = PRN_MONTHLY_TARGET_MONTHS or month_range(START_DATE, END_DATE)

# Source query returns quoted title-case aliases; map them to snake_case before the
# generic normalize/reindex step so they line up with PRNMONTHLY_COLUMNS.
PRNMONTHLY_RENAME_MAP = {
    'Manufacturer ID': 'manufacturer_id',
    'Manufacturer Name': 'manufacturer',
    'Entity': 'entity',
    'Total PRN Quantity': 'total_prn_qty',
}

if should_run('prn_monthly'):
    RUN_RESULTS['prn_monthly'] = run_month_batched(
        label='prn_monthly', sql_template=PRN_MONTHLY_SQL, columns=PRNMONTHLY_COLUMNS,
        parquet_path=PARQUET_FILES['prn_monthly'], target_months=prn_target_months,
        numeric_cols=['total_prn_qty'], date_cols=['target_month'], stamp_col='target_month',
        rename_map=PRNMONTHLY_RENAME_MAP,
    )
else:
    log('SKIPPED (RUN_FLAGS set to No)', label='prn_monthly')


## 19. Query 16 — City list with warehouse-to-city distance & satellite classification

**Not batched** — small city-level reference table (one row per city), run with
`run_single()` (still goes through the same pinned-schema + size-capped part writer as
every other query, so it isn't a special case for the output layer, just for fetching).
Output: `PARQUET_FILES['city_distance']`.

**Classification added, not in the source query — flagging the assumption:** the request
was cities classified as "satellite or not", but the query itself only returns min/max
distance per city — there's no classification column in the SQL. The convention already
used elsewhere in this notebook (IOCC Satellite City Fee, query 3) is `distance_km > 100`
km = satellite. Applied here as **two** derived columns rather than guessing which one is
wanted, since a city can contain multiple dark stores at different distances from their
serving warehouse:
- `is_all_satellite` = `'Yes'` only if **every** dark store in the city is >100km out
  (`BE Min. Distance (km) > 100`)
- `is_any_satellite` = `'Yes'` if **at least one** dark store in the city is >100km out
  (`BE Max. Distance (km) > 100`)

Threshold is `SATELLITE_DISTANCE_KM_THRESHOLD` in CONFIG (default 100, matching query 3).

In [ ]:
CITY_DISTANCE_SQL = r'''WITH Active_assortment AS (
    SELECT
        a.facility_id
    FROM lake_rpc.product_facility_master_assortment a
    WHERE a.active = 1
      AND master_assortment_substate_id IN (1)
      AND a.item_id IN (
            SELECT DISTINCT item_id
            FROM supply_etls.item_details
            WHERE assortment_type = 'Packaged Goods'
              AND handling_type_id <> 8
              AND manufacturer_id <> 1015
      )
    GROUP BY 1
),
ds_be_mapping AS (
    SELECT DISTINCT
        bo.facility_id    AS ds_facility_id,
        bo.be_facility_id AS be_facility_id
    FROM supply_etls.packaged_goods_inventory_and_availability bo
    INNER JOIN Active_assortment aa
        ON aa.facility_id = bo.facility_id
),
distance_calc AS (
    SELECT
        ds_od.city_name,
        111 * (
            ABS(CAST(d.wh_lat AS DOUBLE) - CAST(d.ds_lat AS DOUBLE)) +
            ABS(CAST(d.wh_lon AS DOUBLE) - CAST(d.ds_lon AS DOUBLE))
        ) AS taxi_distance_km
    FROM ds_be_mapping m
    -- Join distance table ONLY on actual BE serving that DS
    JOIN supply_etls.wh_to_ds_distance_v4 d
        ON d.ds_facility_id = m.ds_facility_id
       AND d.wh_facility_id = m.be_facility_id
    -- Get DS city
    JOIN supply_etls.outlet_details ds_od
        ON ds_od.facility_id = m.ds_facility_id
)
SELECT
    city_name AS City,
    ROUND(MIN(taxi_distance_km), 4) AS "BE Min. Distance (km)",
    ROUND(MAX(taxi_distance_km), 4) AS "BE Max. Distance (km)"
FROM distance_calc
GROUP BY 1
ORDER BY 1
'''

print(CITY_DISTANCE_SQL[:400], '...')

In [ ]:
CITYDIST_RENAME_MAP = {
    'City': 'city',
    'BE Min. Distance (km)': 'be_min_distance_km',
    'BE Max. Distance (km)': 'be_max_distance_km',
}


def _add_satellite_classification(df):
    df = df.rename(columns=CITYDIST_RENAME_MAP)
    df['is_all_satellite'] = df['be_min_distance_km'].apply(
        lambda v: 'Yes' if pd.notna(v) and v > SATELLITE_DISTANCE_KM_THRESHOLD else 'No'
    )
    df['is_any_satellite'] = df['be_max_distance_km'].apply(
        lambda v: 'Yes' if pd.notna(v) and v > SATELLITE_DISTANCE_KM_THRESHOLD else 'No'
    )
    return df


if should_run('city_distance'):
    RUN_RESULTS['city_distance'] = run_single(
        label='city_distance', sql_template=CITY_DISTANCE_SQL,
        columns=['City', 'BE Min. Distance (km)', 'BE Max. Distance (km)'],
        parquet_path=PARQUET_FILES['city_distance'],
        numeric_cols=['BE Min. Distance (km)', 'BE Max. Distance (km)'],
        post_process=_add_satellite_classification,
    )
else:
    log('SKIPPED (RUN_FLAGS set to No)', label='city_distance')

## 20. Output Summary — all files produced for this period

Run after the notebook has executed top-to-bottom. Lists every Parquet file (and how
many part files it split into), whether it's
period-scoped, its row count, and on-disk size — one manifest to sanity-check a run.

In [ ]:
_labels_period_scoped = {
    'bcpl_defective_crbs': True, 'recall_assistance': True, 'early_payment': True,
    'fill_rate_penalty': True, 'storage_fees': False, 'rtv_days': False, 'ads_spend': False,
    'turnover_discount': False, 'epod_clause': False, 'sp_benchmarking': False, 'prn_monthly': True,
    'city_distance': False,
}

rows = []
for key, parquet_path in PARQUET_FILES.items():
    if key in RUN_RESULTS:
        r = RUN_RESULTS[key]
        files = r.get('files', [])
        total_mb = sum(os.path.getsize(f) for f in files if os.path.exists(f)) / (1024 * 1024)
        rows.append({
            'key': key, 'base_file': os.path.basename(parquet_path),
            'period_scoped': _labels_period_scoped.get(key),
            'rows': r['rows'], 'n_parts': len(files),
            'total_size_mb': round(total_mb, 2),
            'max_part_size_mb': round(max((os.path.getsize(f) for f in files if os.path.exists(f)), default=0) / (1024 * 1024), 2),
            'status': 'OK',
        })
    elif not should_run(key):
        rows.append({
            'key': key, 'base_file': os.path.basename(parquet_path),
            'period_scoped': _labels_period_scoped.get(key),
            'rows': None, 'n_parts': None, 'total_size_mb': None, 'max_part_size_mb': None,
            'status': 'SKIPPED (RUN_FLAGS = No)',
        })
    else:
        rows.append({
            'key': key, 'base_file': os.path.basename(parquet_path),
            'period_scoped': _labels_period_scoped.get(key),
            'rows': None, 'n_parts': None, 'total_size_mb': None, 'max_part_size_mb': None,
            'status': 'NOT RUN YET (run its section above)',
        })

summary_df = pd.DataFrame(rows)
log(f'Period: {START_DATE} -> {END_DATE}  (only applies to rows where period_scoped=True)')
log(f'Output directory: {OUT_DIR}')
log(f'Max file size: {MAX_FILE_MB} MB per part (see n_parts / max_part_size_mb below)')
log(f"Ran {sum(1 for k in PARQUET_FILES if should_run(k))}/{len(PARQUET_FILES)} queries this pass (see RUN_FLAGS in CONFIG to change)")
display(summary_df)

# Per-query file listing - handy when a query split into multiple parts and you need the
# exact filenames to hand to the reporting layer / glob pattern.
for key, r in RUN_RESULTS.items():
    files = r.get('files', [])
    if len(files) > 1:
        log(f"{len(files)} part files:", label=key)
        for f in files:
            size_mb = os.path.getsize(f) / (1024*1024) if os.path.exists(f) else 0
            log(f'  {os.path.basename(f)}  ({size_mb:.1f} MB)', label=key)
